# Argus — CNN+LSTM Training (Drive-Pull Variant)

A sibling of `11_cnn_lstm_training_upload.ipynb`. Everything downstream of setup — the split,
the `tf.data` pipeline, both model variants, training, evaluation — is identical to `11`. The
only thing that differs is **how the window index CSV and the face crops get onto the Colab
VM**: `11` takes them via `files.upload()` (a browser upload, redone every session); this
notebook **copies them once from a Google Drive folder** onto the Colab VM's local disk, then
works entirely off local disk from there.

This keeps `11`'s core fix — never let the `tf.data` image pipeline read thousands of small
JPEGs over Drive's FUSE mount, which is what triggered `Input/output error` / `A Google Drive
timeout has occurred` in `10` — while dropping `11`'s friction of re-uploading a ~2 GB zip by
hand every runtime. One big sequential file copy off Drive has none of the per-file overhead or
rate-limiting of many small concurrent reads.

**What you need on Drive before running**, under
`/content/drive/MyDrive/Argus/dataset/dataset_processed/` (the `DRIVE_PROCESSED_DIR` config in
the setup cell — edit it if yours differs):
- `cnn_lstm_windows_index_forcolab.csv` — the window index from
  `09_dataset_creation_cnn_lstm.ipynb` (the `_forcolab` variant, with basename-only crop paths).
- `face_crops.zip` — a zip of the `face_crops/` folder `06_dataset_creation_face_crops.ipynb`
  produced.
- `best_cnn_scratch_face_crops.keras` — `07_cnn_training.ipynb`'s trained checkpoint, in
  `/content/drive/MyDrive/Argus/models/` (only needed for the Frozen-CNN-Embedding variant; if
  `07`'s own run already left it there, the cell for it picks it up automatically).

**Drive is also mounted for `models_folder`**, exactly as in `11`: checkpoints save straight to
Drive as training happens, so a mid-training disconnect never loses the best-so-far model.

## Google Drive Connection & Project Setup


In [ ]:
import os
import shutil
import datetime
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

# --- Drive layout (matches 09/10's convention; edit if yours differs) ---
PROJECT_FOLDER = "/content/drive/MyDrive/Argus"
models_folder = f"{PROJECT_FOLDER}/models"
DRIVE_PROCESSED_DIR = f"{PROJECT_FOLDER}/dataset/dataset_processed"
os.makedirs(models_folder, exist_ok=True)

# The two artifacts you uploaded to DRIVE_PROCESSED_DIR outside Colab.
CSV_NAME = "cnn_lstm_windows_index_forcolab.csv"
ZIP_NAME = "face_crops.zip"

# --- Copy the window index CSV from Drive to local disk, then read the LOCAL copy ---
# One sequential file copy off Drive is fine -- it's the many-small-concurrent-reads pattern
# (the per-crop JPEG decode in the tf.data pipeline below) that was unreliable, not this.
_drive_csv = os.path.join(DRIVE_PROCESSED_DIR, CSV_NAME)
cnn_lstm_windows_csv_path = f"/content/{CSV_NAME}"
if not os.path.exists(_drive_csv):
    raise FileNotFoundError(
        f"'{_drive_csv}' not found on Drive. Upload {CSV_NAME} (from "
        "09_dataset_creation_cnn_lstm.ipynb) into DRIVE_PROCESSED_DIR, or fix "
        "DRIVE_PROCESSED_DIR / CSV_NAME above."
    )
if not os.path.exists(cnn_lstm_windows_csv_path):
    print(f"Copying {_drive_csv} -> {cnn_lstm_windows_csv_path} ...")
    shutil.copy(_drive_csv, cnn_lstm_windows_csv_path)
else:
    print(f"{cnn_lstm_windows_csv_path} already on local disk -- using it.")

df_cnn_lstm_windows = pd.read_csv(cnn_lstm_windows_csv_path)
if 'geometric_feature_seq' not in df_cnn_lstm_windows.columns:
    raise ValueError(
        "cnn_lstm_windows_index.csv has no 'geometric_feature_seq' column -- re-run "
        "09_dataset_creation_cnn_lstm.ipynb (it now also extracts per-frame EAR/MAR/"
        "blendshape features for fusion; a CSV from before that update won't have this "
        "column)."
    )
print(f"Loaded window index: {len(df_cnn_lstm_windows)} windows, "
      f"{df_cnn_lstm_windows['subject'].nunique()} subjects.")

# --- Model Management Configuration ---
FORCE_RETRAIN = True
VERSION_STR = datetime.datetime.now().strftime("%Y%m%d_%H%M")
cnn_lstm_scratch_model_path = os.path.join(models_folder, f"cnn_lstm_scratch_{VERSION_STR}.keras")
cnn_lstm_mobilenet_model_path = os.path.join(models_folder, f"cnn_lstm_mobilenet_{VERSION_STR}.keras")

def get_latest_model(folder, prefix, extension):
    """Return the path of the most recently versioned model matching prefix/extension, or None."""
    if not os.path.exists(folder):
        return None
    matches = [f for f in os.listdir(folder) if f.startswith(prefix) and f.endswith(extension)]
    if not matches:
        return None
    return os.path.join(folder, sorted(matches)[-1])

print(f"✅ Model management initialized. Current version: {VERSION_STR}. Force retrain: {FORCE_RETRAIN}")

## Splitting the Dataset

Group-aware split by subject, same rationale and same rule as `07_cnn_training.ipynb`'s
now-migrated `split_by_subject_fold` — ported here and called on the window index instead of the
crop index (same `subject`/`level` columns, same meaning). Binary task (`Not Drowsy` / `Drowsy`):
only subjects with **both** classes represented among their windows are eligible to be a held-out
*test* or *validation* subject; genuinely single-class subjects (e.g. a failed clip extraction)
still contribute real training windows for whichever class they have, they just can't be a
held-out fold. The split happens on the **index CSV** (window metadata + `;`-joined path lists),
before any image is decoded — pixel loading happens lazily in the `tf.data` pipeline below, same
as 07.

**Validation is now a real KFold fold, not the incomplete-subject pool.** The 3-class version of
this notebook derived `val` from the pool of class-incomplete subjects, and always put every such
subject into validation without ever rotating them. Under binary labels almost every UTA-RLDD
subject has both classes (each recorded an alert clip and a drowsy clip), so that pool is usually
empty — `val` is now fold `(fold_idx + 1) % n_splits` of the same `StratifiedGroupKFold`, a
distinct, disjoint subject group from the test fold. It's still small (one KFold fold's worth of
subjects) and still a plausible independent source of the epoch-to-epoch `val_macro_f1` noise
seen in training logs. `split_by_subject_fold` stays parameterized by `fold_idx`/`n_splits`
specifically so a future subject-fold cross-validation diagnostic (mirroring
`07_cnn_training.ipynb`'s `RUN_CROSS_VALIDATION` cells) could measure how much that noise
actually moves the reported number.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
import numpy as np


def split_by_subject_fold(df, fold_idx=0, n_splits=None, random_state=42):
    """Subject-grouped train/val/test split for the window index, parameterized by fold index so
    the same logic can drive both the main fold-0 split below and a future subject-fold
    cross-validation diagnostic, without duplicating the split rules. Ported from
    07_cnn_training.ipynb's now-migrated version -- called on the window index instead of the
    crop index (same `subject`/`level` columns, same meaning).

    Binary task (Not Drowsy / Drowsy). A subject is "complete" if it has windows for BOTH classes
    {1, 2}; only complete subjects can be held out as *test* or *validation* subjects, since a
    single-class held-out fold can't score both classes. Incomplete subjects (only one class
    present -- e.g. a failed clip extraction) are NOT dropped: they stay in the training pool as
    valid windows for whichever class they have.

        test  = fold `fold_idx` of a StratifiedGroupKFold over the complete subjects
        val   = fold `(fold_idx + 1) % n_splits` -- a different, disjoint subject group
        train = every other complete subject + all incomplete subjects

    The 3-class version of this notebook derived `val` from the pool of class-incomplete subjects
    instead. Under binary labels almost every UTA-RLDD subject has both classes (each recorded an
    alert clip and a drowsy clip), so that pool is usually empty -- hence val is now a KFold fold
    like test.

    Returns (df_train, df_val, df_test, complete_subjects, incomplete_subjects, n_splits_used).
    """
    subject_level_sets = df.groupby('subject')['level'].agg(set)
    complete_subjects = subject_level_sets[subject_level_sets == {1, 2}].index
    incomplete_subjects = subject_level_sets.index.difference(complete_subjects)

    if len(complete_subjects) < 3:
        raise ValueError(
            f"Only {len(complete_subjects)} subject(s) have both classes ({list(complete_subjects)}) "
            "-- need at least 3 to hold one out for testing and one for validation. Finish dataset "
            "extraction (06_dataset_creation_face_crops.ipynb + 09_dataset_creation_cnn_lstm.ipynb) "
            "for more subjects before training."
        )

    df_complete = df[df['subject'].isin(complete_subjects)]
    n_splits = n_splits or min(5, len(complete_subjects))
    if not (0 <= fold_idx < n_splits):
        raise ValueError(f"fold_idx={fold_idx} out of range for n_splits={n_splits}")

    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_subjects = [
        set(df_complete.iloc[test_pos]['subject'])
        for _, test_pos in sgkf.split(df_complete, df_complete['level'], groups=df_complete['subject'])
    ]

    test_subjects = fold_subjects[fold_idx]
    val_subjects = fold_subjects[(fold_idx + 1) % n_splits]

    test_mask = df['subject'].isin(test_subjects)
    val_mask = df['subject'].isin(val_subjects)
    train_mask = ~test_mask & ~val_mask

    df_train = df[train_mask].reset_index(drop=True)
    df_val = df[val_mask].reset_index(drop=True)
    df_test = df[test_mask].reset_index(drop=True)

    all_levels = set(df['level'].unique())
    for name, part in (('Validation', df_val), ('Test', df_test)):
        missing = all_levels - set(part['level'].unique())
        if missing:
            raise ValueError(f"{name} fold missing level(s) {sorted(missing)} -- treat as a bug.")

    return df_train, df_val, df_test, complete_subjects, incomplete_subjects, n_splits


df_train, df_val, df_test, complete_subjects, incomplete_subjects, N_SPLITS = split_by_subject_fold(
    df_cnn_lstm_windows, fold_idx=0
)
print(f"Complete-class subjects ({len(complete_subjects)}): {list(complete_subjects)}")
print(f"Incomplete subjects ({len(incomplete_subjects)}): {list(incomplete_subjects)}")
print(f"Train: {len(df_train)} windows, Val: {len(df_val)} windows, Test: {len(df_test)} windows")
print(f"Test-fold windows by duration:\n{df_test['window_duration_sec'].value_counts().sort_index()}")

## Copying Face Crops from Drive

`10_cnn_lstm_training.ipynb` let the `tf.data` pipeline read each crop JPEG straight off Drive's
FUSE mount -- thousands of small concurrent reads, which is exactly what threw `Input/output
error` / `A Google Drive timeout has occurred`. `11_cnn_lstm_training_upload.ipynb` fixed that
by uploading a zip by hand every session. This notebook does the same thing without the manual
step: copy the single `face_crops.zip` off Drive (one big sequential transfer -- none of the
per-file overhead) onto the Colab VM's local disk, unzip it there, and point every `image_paths`
entry at the local copy. Training then never touches Drive for image reads.

In [ ]:
import os
import shutil
import zipfile

LOCAL_FACE_CROPS_DIR = "/content/face_crops_local"
os.makedirs(LOCAL_FACE_CROPS_DIR, exist_ok=True)

_drive_zip = os.path.join(DRIVE_PROCESSED_DIR, ZIP_NAME)
_local_zip = f"/content/{ZIP_NAME}"
if not os.path.exists(_local_zip):
    if not os.path.exists(_drive_zip):
        raise FileNotFoundError(
            f"'{_drive_zip}' not found on Drive. Zip the face_crops/ folder "
            "(06_dataset_creation_face_crops.ipynb's output) outside Colab, upload it into "
            "DRIVE_PROCESSED_DIR, or fix DRIVE_PROCESSED_DIR / ZIP_NAME in the setup cell."
        )
    print(f"Copying {_drive_zip} -> {_local_zip} (one sequential transfer off Drive) ...")
    shutil.copy(_drive_zip, _local_zip)
else:
    print(f"{_local_zip} already on local disk -- using it.")

print(f"Unzipping {_local_zip} -> {LOCAL_FACE_CROPS_DIR} ...")
with zipfile.ZipFile(_local_zip, 'r') as zf:
    zf.extractall(LOCAL_FACE_CROPS_DIR)

# The zip may have preserved a folder structure (e.g. face_crops/subject_..._sN.jpg) instead
# of a flat one -- flatten it here so LOCAL_FACE_CROPS_DIR always holds the images directly,
# matching what _localize_image_paths below assumes.
for _root, _dirs, _filenames in os.walk(LOCAL_FACE_CROPS_DIR):
    for _fn in _filenames:
        _src = os.path.join(_root, _fn)
        _dst = os.path.join(LOCAL_FACE_CROPS_DIR, _fn)
        if _src != _dst:
            os.replace(_src, _dst)

_n_local = len([f for f in os.listdir(LOCAL_FACE_CROPS_DIR)
                if os.path.isfile(os.path.join(LOCAL_FACE_CROPS_DIR, f))])
print(f"Done: {_n_local} files at {LOCAL_FACE_CROPS_DIR}.")


def _localize_image_paths(image_paths_str: str) -> str:
    """Rewrites a ';'-joined string of crop paths (as written by
    09_dataset_creation_cnn_lstm.ipynb, originally pointing at Drive) to the local copy
    instead -- matched by filename only, not by prefix-replacing the original Drive path,
    since that path never existed in this notebook's session at all.
    """
    return ";".join(
        os.path.join(LOCAL_FACE_CROPS_DIR, os.path.basename(p)) for p in image_paths_str.split(";")
    )


# df_cnn_lstm_windows itself needs this too, not just the three split copies below -- the
# frozen-embedding section's embedding precompute cell reads df_cnn_lstm_windows['image_paths']
# directly (it needs every unique crop across train+val+test, not just df_train's), and
# df_train/df_val/df_test are independent copies (`.reset_index(drop=True)` after the split),
# so localizing only those three left df_cnn_lstm_windows's paths stale.
for _df in (df_cnn_lstm_windows, df_train, df_val, df_test):
    _df["image_paths"] = _df["image_paths"].apply(_localize_image_paths)

_sample_path = df_train["image_paths"].iloc[0].split(";")[0]
assert _sample_path.startswith(LOCAL_FACE_CROPS_DIR), "image_paths wasn't localized -- check the unzip above."
assert os.path.exists(_sample_path), f"{_sample_path} not found locally -- check the zip contains this file."
print(f"df_train/df_val/df_test image_paths now point at {LOCAL_FACE_CROPS_DIR}.")

## Building the `tf.data` Pipeline

Each row's `image_paths` (`;`-joined) has exactly `n_real_frames` real images (3/5/10/20 — no
partial windows, see `09`). Per window:

1. Split the path list, decode + resize each real frame.
2. Zero-**pre**-pad to `MAX_TIMESTEPS_IMG=20` (zeros first, real frames last) — matching this
   project's established rolling-buffer convention (root `CLAUDE.md`: padding has to match how a
   live buffer actually fills, oldest-dropped/newest-appended).
3. Build a boolean mask of shape `(MAX_TIMESTEPS_IMG,)` **directly from the already-known
   `n_real_frames` value**, not by inferring it from all-zero pixels. A plain Keras `Masking`
   layer only reduces over the *last* axis of its input — that's exactly right for the geometric
   LSTM's all-zero 58-float timestep, but wrong for an all-zero `(H, W, 3)` image timestep here.
   Computing the mask from the known real length sidesteps that entirely, and is strictly more
   correct anyway: a genuinely all-black *real* frame would never be misdetected as padding this
   way, unlike with a value-based `Masking` layer.
4. Feed the mask directly into the `LSTM` layer's `call()` (`LSTM(...)(x, mask=mask_input)`) —
   the officially-supported way to pass an explicit mask to a Keras RNN layer — rather than a
   preceding `Masking` layer.

**Window-level augmentation, not per-frame.** Every frame in a window is the same continuous
stretch of one clip, so flip/rotation/zoom/contrast/brightness must be sampled **once per
window** and applied identically to every real frame in it — flipping frame 3 but not frame 7 of
the same window would be incoherent. Calling `tf.keras.layers.Random*` directly on a
`(T, H, W, C)` tensor would sample independently per leading-axis entry (its normal batch
behavior), reproducing exactly that wrong per-frame-independent behavior — so rotation/zoom/
contrast are applied via a fold-time-into-channels trick instead (`(T,H,W,3) → (1,H,W,T*3)`,
augment once, unfold), and flip/brightness via one manually-sampled decision applied to the whole
window at once.

**Per-frame Gaussian noise is the one exception — applied independently per real frame,
deliberately.** Real camera sensor noise genuinely varies frame to frame in actual video, unlike
flip/rotation/zoom (properties of the whole camera framing), so sampling it independently per
frame is the *realistic* choice here, not a bug to avoid.

**`BATCH_SIZE=8` / `BATCH_SIZE_MOBILENET=4`, both down from 07's 32.** Each padded scratch-CNN
example is `20×96×96×3` floats — roughly 20x heavier than one of 07's single images — and the
MobileNetV2 variant's larger `128×128` frames make its examples heavier still, so both batch
sizes are reduced accordingly rather than left at 07's value.


**Update: a parallel geometric feature sequence.** Each row's `geometric_feature_seq`
(`;`-per-frame, `,`-per-feature, written by `09_dataset_creation_cnn_lstm.ipynb`) is parsed and
zero-*pre*-padded the same way as `image_paths`, producing a `(MAX_TIMESTEPS_IMG,
NUM_GEO_FEATURES)` tensor that lines up 1:1 against the same `mask` used for the image sequence.
This does **not** go through the image augmentation pipeline -- see "Model Definition" below for
where it's normalized and fused with the CNN embedding.


In [ ]:
import tensorflow as tf
import numpy as np
import os

os.makedirs('/content/tf_cache', exist_ok=True)  # ds.cache() below needs this directory to already exist -- it won't create it, and writing the cache lockfile fails with a NotFoundError if it's missing.

MAX_TIMESTEPS_IMG = 100                # Must match 09_dataset_creation_cnn_lstm.ipynb's own
                                        # MAX_TIMESTEPS_IMG (max_context_sec * sampling_fps = 20*5 = 100).
IMG_SIZE = 96                          # From-scratch CNN input resolution -- same as 07 (same
                                        # underlying crops, no reason to change it here).
IMG_SIZE_MOBILENET = 128               # Deliberately larger than 96 -- see the Model section
                                        # below for why (07's alpha=0.35/96x96 collapse).
BATCH_SIZE = 8                         # Down from 07's 32 -- see markdown above.
BATCH_SIZE_MOBILENET = 4               # Smaller still: larger resolution + a heavier backbone.
CLASS_NAMES = ['Not Drowsy', 'Drowsy']

# --- Minority-class balancing (see the training markdown) ---
# True: draw the from-scratch training set 50/50 from the two classes so every BATCH_SIZE=8
# batch has a stable mix (tf.data sample_from_datasets over two per-class .repeat() streams),
# and set class_weight=None in the training cell. The 2:1 Not Drowsy : Drowsy imbalance is mild,
# but at batch size 8 a single class_weight-boosted example can dominate a batch -- resampling
# sidesteps that. The final Drowsy operating point is set by the decision-threshold cell after
# evaluation, not by this.
USE_BALANCED_SAMPLING = True
NOISE_STDDEV = 0.03                    # Std-dev of additive Gaussian noise on a [0,255]-scaled
                                        # image (applied as NOISE_STDDEV * 255) -- a conservative
                                        # starting point; sanity-check visually before trusting it
                                        # (see "Augmented Sample Preview" below), same bar 07's own
                                        # augmentation choices were held to.

# Must match 09_dataset_creation_cnn_lstm.ipynb's GEO_FEATURE_NAMES exactly, same order -- these
# just name what each column of a parsed geometric_feature_seq frame actually is.
GEO_FEATURE_NAMES = [
    'EAR_left', 'EAR_right', 'MAR',
    'eyeWideRight', 'eyeBlinkRight', 'browOuterUpRight',
    'eyeBlinkLeft', 'eyeSquintLeft', 'eyeWideLeft', 'eyeSquintRight',
]
NUM_GEO_FEATURES = len(GEO_FEATURE_NAMES)


def _decode_and_resize(path, img_size):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_jpeg(image_bytes, channels=3)
    image = tf.image.resize(image, [img_size, img_size])
    # Cast back to uint8 here rather than leaving float32: this is the tensor that gets cached to
    # disk below (via make_dataset's ds.cache(...)), and uint8 is 4x smaller on disk than float32
    # for the exact same pixel data. The cast back to float32 the model/augmentation actually need
    # happens once, downstream, after the cache read -- see make_dataset.
    return tf.cast(image, tf.uint8)


def _parse_geo_seq(geo_seq_str, num_geo_features):
    """'v1,...,v10;v1,...,v10;...' (one ';'-joined group of NUM_GEO_FEATURES comma-joined floats
    per real frame, written by 09_dataset_creation_cnn_lstm.ipynb) -> (n_real_frames,
    num_geo_features) float32 tensor.
    """
    frames = tf.strings.split(geo_seq_str, ';')
    per_frame = tf.strings.split(frames, ',').to_tensor(default_value='0.0', shape=[None, num_geo_features])
    return tf.strings.to_number(per_frame, out_type=tf.float32)


def load_window(image_paths_str, geo_seq_str, n_real_frames, label, img_size, max_timesteps,
                 num_geo_features):
    """One window row -> (padded_images, padded_geo, mask, label).

    `n_real_frames` is the exact, already-known count of real frames (not inferred). Padding is
    zero-*pre*-padded (zeros first, real frames last) to match this project's rolling-buffer
    convention -- applied identically to the image sequence and the geometric feature sequence,
    so both line up against the same `mask`. `padded_images` is uint8 here (see
    _decode_and_resize) -- callers that need float32 (augmentation, the model) cast after this.
    """
    paths = tf.strings.split(image_paths_str, ';')
    real_images = tf.map_fn(
        lambda p: _decode_and_resize(p, img_size), paths, fn_output_signature=tf.uint8,
    )  # (n_real_frames, img_size, img_size, 3), uint8
    real_geo = _parse_geo_seq(geo_seq_str, num_geo_features)  # (n_real_frames, num_geo_features)

    pad_amount = max_timesteps - n_real_frames
    padded_images = tf.pad(real_images, [[pad_amount, 0], [0, 0], [0, 0], [0, 0]])
    padded_images.set_shape([max_timesteps, img_size, img_size, 3])
    padded_geo = tf.pad(real_geo, [[pad_amount, 0], [0, 0]])
    padded_geo.set_shape([max_timesteps, num_geo_features])

    mask = tf.range(max_timesteps) >= pad_amount  # True for the last n_real_frames positions

    return padded_images, padded_geo, mask, label


# Same rotation/zoom/contrast augmenter as 07's geometric_augmenter, reused via the fold/unfold
# trick below so ONE random draw covers the whole window instead of one draw per frame.
geometric_augmenter = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.05),   # ~+/-18 degrees, same as 07
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.2),
])


def _apply_shared_geometric_augmentation(images, img_size):
    """Applies `geometric_augmenter` with ONE shared random draw across every frame in the
    window. Keras preprocessing layers sample an independent transform per element of whatever
    leading axis they're given -- calling them directly on images shaped (T, H, W, 3) would
    reproduce 07's independent-per-image behavior, wrong for a sequence. Folding time into the
    channel axis first -- (T, H, W, 3) -> (1, H, W, T*3) -- makes the layer see one single image
    with many channels instead of T separate images, so it draws one transform for the whole
    thing; unfolding afterward restores the per-frame tensor.
    """
    t = tf.shape(images)[0]
    folded = tf.reshape(tf.transpose(images, [1, 2, 0, 3]), [1, img_size, img_size, -1])
    folded = geometric_augmenter(folded, training=True)
    unfolded = tf.reshape(folded, [img_size, img_size, t, 3])
    return tf.transpose(unfolded, [2, 0, 1, 3])


def augment_window(images, geo, mask, label, img_size):
    """Augmentation shared across every real frame in the window (flip/brightness/rotation/zoom/
    contrast), plus independent per-frame Gaussian noise -- see the markdown above for why noise
    is the one exception applied per-frame rather than per-window. `geo` (the geometric feature
    sequence) deliberately passes through untouched -- these image-space transforms have no
    corresponding effect to apply to an already-computed EAR/MAR/blendshape ratio, unlike
    07/09's crop pipeline, which augments before feature extraction ever happens. `images` must
    already be float32 by the time it reaches here (see make_dataset's cast-after-cache step) --
    the arithmetic below (+delta, +noise) doesn't work against the uint8 tensor load_window hands
    back.
    """
    # --- Flip: one shared coin flip for the whole window ---
    do_flip = tf.random.uniform([]) < 0.5
    images = tf.cond(do_flip, lambda: tf.image.flip_left_right(images), lambda: images)

    # --- Brightness: one shared delta, added identically to every frame ---
    brightness_delta = tf.random.uniform([], -0.15, 0.15) * 255.0
    images = tf.clip_by_value(images + brightness_delta, 0.0, 255.0)

    # --- Rotation/zoom/contrast: one shared draw via the fold-into-channels trick ---
    images = _apply_shared_geometric_augmentation(images, img_size)

    # --- Per-frame Gaussian noise: independent per real timestep, deliberately ---
    noise = tf.random.normal(tf.shape(images), stddev=NOISE_STDDEV * 255.0)
    images = tf.clip_by_value(images + noise, 0.0, 255.0)

    return images, geo, mask, label


def make_dataset(df, training: bool, img_size: int, batch_size: int, cache_name: str = None,
                 balanced: bool = False):
    """Builds one split's tf.data pipeline.

    `cache_name`, when given, caches the decoded-but-still-uint8 window (post-load_window,
    pre-float32-cast) to LOCAL Colab disk under /content/tf_cache/ -- NOT to Drive. The expensive
    part of this pipeline is the per-window JPEG decode over Drive's FUSE mount (up to
    MAX_TIMESTEPS_IMG images per window); caching right after decode, before the float32 cast,
    keeps on-disk size to ~img_size^2*3 bytes/frame (uint8) instead of 4x that (float32). Note:
    /content/ is the Colab VM's ephemeral disk -- this speeds up iteration within one session,
    not across sessions, and a full pass must complete once to populate the cache.

    `balanced` (training only): draw the training set 50/50 from the two classes. Each class gets
    its OWN pipeline with its OWN cache file (`<cache_name>_neg` / `_pos`) -- filtering a single
    shared file-backed cache from two concurrent iterators is unsafe -- then the two infinite,
    independently-reshuffled per-class streams are interleaved with sample_from_datasets. The
    result is infinite, so .fit() must pass an explicit steps_per_epoch.
    """
    def _pipeline(df_slice, cname):
        image_paths = df_slice['image_paths'].to_numpy()
        geo_seqs = df_slice['geometric_feature_seq'].to_numpy()
        n_real_frames = df_slice['n_real_frames'].to_numpy(dtype=np.int32)
        labels = (df_slice['level'].to_numpy(dtype=np.int32) - 1)  # 0=Not Drowsy, 1=Drowsy
        d = tf.data.Dataset.from_tensor_slices((image_paths, geo_seqs, n_real_frames, labels))
        d = d.map(
            lambda p, g, n, l: load_window(p, g, n, l, img_size, MAX_TIMESTEPS_IMG, NUM_GEO_FEATURES),
            num_parallel_calls=tf.data.AUTOTUNE,
        )
        if cname is not None:
            d = d.cache(f'/content/tf_cache/{cname}')
        d = d.map(lambda img, g, m, l: (tf.cast(img, tf.float32), g, m, l),
                  num_parallel_calls=tf.data.AUTOTUNE)
        return d

    if training and balanced:
        neg = _pipeline(df[df['level'] == 1], f'{cache_name}_neg' if cache_name else None)
        pos = _pipeline(df[df['level'] == 2], f'{cache_name}_pos' if cache_name else None)
        neg = neg.repeat().shuffle(256, seed=42)
        pos = pos.repeat().shuffle(256, seed=43)
        ds = tf.data.Dataset.sample_from_datasets([neg, pos], weights=[0.5, 0.5], seed=42)
        ds = ds.map(lambda img, g, m, l: augment_window(img, g, m, l, img_size),
                    num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = _pipeline(df, cache_name)
        if training:
            ds = ds.shuffle(buffer_size=2048, seed=42)
            ds = ds.map(lambda img, g, m, l: augment_window(img, g, m, l, img_size),
                        num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.map(lambda img, g, m, l: ((img, g, m), l), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


# --- From-scratch CNN datasets (IMG_SIZE=96) -- cached to local disk, see make_dataset above ---
train_ds = make_dataset(df_train, training=True, img_size=IMG_SIZE, batch_size=BATCH_SIZE,
                         cache_name='train_96', balanced=USE_BALANCED_SAMPLING)
val_ds = make_dataset(df_val, training=False, img_size=IMG_SIZE, batch_size=BATCH_SIZE,
                       cache_name='val_96')
test_ds = make_dataset(df_test, training=False, img_size=IMG_SIZE, batch_size=BATCH_SIZE,
                        cache_name='test_96')

# --- MobileNetV2 datasets (IMG_SIZE_MOBILENET=128) -- a separate pipeline, not a resize of the
# above, since the two backbones deliberately use different input resolutions (see Model below).
# Deliberately left uncached: MobileNetV2 is currently disabled in this notebook (see
# CLAUDE.md's "What we found" -- severe overfitting on the last run) and at 128px would add
# roughly another 20GB+ of local disk on top of the ~16GB above for a backbone not currently
# being trained. Pass cache_name='train_128'/'val_128'/'test_128' back in here if that changes.
train_ds_mobilenet = make_dataset(df_train, training=True, img_size=IMG_SIZE_MOBILENET,
                                   batch_size=BATCH_SIZE_MOBILENET)
val_ds_mobilenet = make_dataset(df_val, training=False, img_size=IMG_SIZE_MOBILENET,
                                 batch_size=BATCH_SIZE_MOBILENET)
test_ds_mobilenet = make_dataset(df_test, training=False, img_size=IMG_SIZE_MOBILENET,
                                  batch_size=BATCH_SIZE_MOBILENET)

print(f"IMG_SIZE={IMG_SIZE}, BATCH_SIZE={BATCH_SIZE}, balanced_sampling={USE_BALANCED_SAMPLING} | "
      f"IMG_SIZE_MOBILENET={IMG_SIZE_MOBILENET}, BATCH_SIZE_MOBILENET={BATCH_SIZE_MOBILENET} | "
      f"NUM_GEO_FEATURES={NUM_GEO_FEATURES}")
print("train_ds/val_ds/test_ds cache their decoded (uint8) windows under /content/tf_cache/ -- "
      "the first full pass through each pays the Drive-decode cost once per Colab runtime; every "
      "pass after reads local disk instead.")


### Augmented Sample Preview

A cheap, direct sanity check before trusting the pipeline blind: pull one augmented training
window and confirm (a) the geometric augmentation (flip/rotation/zoom/contrast) looks identical
across every real frame, (b) the per-frame noise is visibly present but not so strong it obscures
the eye/mouth region, and (c) the mask's real-frame count matches `n_real_frames`, and the real
frames are the *last* positions (pre-padding), not the first.


In [ ]:
import matplotlib.pyplot as plt

# Built directly from a single row instead of train_ds.take(1): train_ds shuffles with
# buffer_size=2048 (see make_dataset above), so a plain .take(1) on it forces TF to decode
# ~2048 windows' worth of Drive-mounted JPEGs -- up to MAX_TIMESTEPS_IMG images each -- just to
# hand back one. That's the same shuffle-buffer-fill cost behind epoch 1's ~2081s vs every later
# epoch's ~206s. This preview only needs one real, augmented window, so it decodes exactly one,
# reusing load_window/augment_window directly rather than a copy of their logic.
preview_row = df_train.iloc[[0]]
preview_ds = tf.data.Dataset.from_tensor_slices((
    preview_row['image_paths'].to_numpy(),
    preview_row['geometric_feature_seq'].to_numpy(),
    preview_row['n_real_frames'].to_numpy(dtype=np.int32),
    (preview_row['level'].to_numpy(dtype=np.int32) - 1),
))
preview_ds = preview_ds.map(
    lambda p, g, n, l: load_window(p, g, n, l, IMG_SIZE, MAX_TIMESTEPS_IMG, NUM_GEO_FEATURES))
preview_ds = preview_ds.map(lambda img, g, m, l: (tf.cast(img, tf.float32), g, m, l))
preview_ds = preview_ds.map(lambda img, g, m, l: augment_window(img, g, m, l, IMG_SIZE))
preview_ds = preview_ds.map(lambda img, g, m, l: ((img, g, m), l))  # match make_dataset's final
                                                                     # packaging step -- missed
                                                                     # this the first time, which
                                                                     # is exactly why the unpack
                                                                     # below saw 4 values, not 2.
preview_ds = preview_ds.batch(1)

for (sample_images, sample_geo, sample_mask), sample_label in preview_ds.take(1):
    break

sample_idx = 0
images_np = sample_images[sample_idx].numpy()
geo_np = sample_geo[sample_idx].numpy()
mask_np = sample_mask[sample_idx].numpy()
n_real = int(mask_np.sum())

print(f"Label: {CLASS_NAMES[int(sample_label[sample_idx].numpy())]}, "
      f"mask real-frame count: {n_real} (real frames occupy the LAST {n_real} of "
      f"{MAX_TIMESTEPS_IMG} positions -- pre-padding)")
assert (mask_np[-n_real:]).all() and not (mask_np[:MAX_TIMESTEPS_IMG - n_real]).any(), \
    "Mask shape doesn't match the expected pre-padding layout -- check load_window's padding order."

# The geometric feature sequence should be zero for every padded (non-real) position and
# non-trivial for every real one -- a direct check that its padding lines up with the image mask.
geo_real = geo_np[-n_real:]
geo_padded = geo_np[:MAX_TIMESTEPS_IMG - n_real]
assert not geo_padded.any(), "Padded geometric feature positions should be all-zero."
print(f"First real frame's geometric features ({GEO_FEATURE_NAMES}):\n{geo_real[0].round(4)}")

real_frames = images_np[MAX_TIMESTEPS_IMG - n_real:]
n_preview = min(8, len(real_frames))
preview_idx = np.linspace(0, len(real_frames) - 1, n_preview, dtype=int)
fig, axes = plt.subplots(1, n_preview, figsize=(2 * n_preview, 2.5))
for ax, idx in zip(np.atleast_1d(axes), preview_idx):
    ax.imshow(np.clip(real_frames[idx] / 255.0, 0, 1))
    ax.set_title(f"t={idx}")
    ax.axis('off')
plt.suptitle("Augmented training window -- geometric transform should look identical across "
             "frames; noise should vary frame to frame")
plt.tight_layout()
plt.show()


## Loss Function

`SparseCategoricalFocalLoss`, ported unchanged from `07_cnn_training.ipynb` (same reasoning:
`Drowsy` is plausibly the rarest, most safety-critical class, and focal loss down-weights
easy/already-confident examples so training focuses on the harder, more ambiguous ones).


In [ ]:
from tensorflow.keras.losses import SparseCategoricalCrossentropy, CategoricalCrossentropy
import tensorflow as tf
import keras

num_classes = len(CLASS_NAMES)  # CLASS_NAMES defined earlier in "Building the tf.data Pipeline"

USE_FOCAL_LOSS = True
FOCAL_GAMMA = 2.0
LABEL_SMOOTHING = 0.05

@keras.saving.register_keras_serializable(package="CustomLosses")
class SparseCategoricalFocalLoss(keras.losses.Loss):
    def __init__(self, gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING,
                 reduction='sum_over_batch_size', name='sparse_categorical_focal_loss'):
        super().__init__(reduction=reduction, name=name)
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.cce_per_example = CategoricalCrossentropy(from_logits=False, reduction=None)

    def call(self, y_true, y_pred):
        y_true_int = tf.cast(y_true, tf.int32)
        num_classes_tensor = tf.shape(y_pred)[1]
        y_true_one_hot = tf.one_hot(y_true_int, depth=num_classes_tensor)

        if self.label_smoothing > 0.0:
            num_classes_float = tf.cast(num_classes_tensor, tf.float32)
            y_true_one_hot = y_true_one_hot * (1.0 - self.label_smoothing) + (self.label_smoothing / num_classes_float)

        ce = self.cce_per_example(y_true_one_hot, y_pred)
        probs_true_class = tf.reduce_sum(y_pred * y_true_one_hot, axis=-1)
        probs_true_class = tf.clip_by_value(probs_true_class, 1e-7, 1.0 - 1e-7)

        modulating_factor = tf.pow(1.0 - probs_true_class, self.gamma)
        return modulating_factor * ce

    def get_config(self):
        config = super().get_config()
        config.update({'gamma': self.gamma, 'label_smoothing': self.label_smoothing})
        return config


def build_loss():
    if USE_FOCAL_LOSS:
        return SparseCategoricalFocalLoss(gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING)
    return SparseCategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)


## Model Definition — `TimeDistributed(CNN) → LSTM`


> **Correction (learning-rate bug + regularization revert).** An earlier drive-pull edit set
> `lr_schedule_scratch`'s `initial_learning_rate` to `1e-8` and `build_frozen_embedding_lstm`'s
> default `learning_rate` to `1e-8` (never overridden at the call site). Both models therefore
> trained at ~`1e-8` and *did not learn* — the from-scratch model's train accuracy sat flat at
> ~0.50, and the frozen-embedding model's `val_accuracy` / `val_macro_f1` were frozen at their
> untrained-init values (`0.7511` / `0.7151`) for the whole run. That run's "still overfitting
> after round-2 regularization" reading was not real. The "round 2" bump (`WEIGHT_DECAY`
> `1e-4→3e-4`, LSTM `dropout`/`recurrent_dropout` `0.3→0.4`, head `Dropout` `0.5→0.6`) has been
> **reverted** to the round-1 set that `11_cnn_lstm_training_upload.ipynb` uses, alongside the
> LR fix (`initial_learning_rate=1e-4`, frozen builder `learning_rate=1e-3`). Re-raise
> regularization only if a genuinely-training run overfits. The `CosineDecayRestarts` /
> `AdamW` / `recurrent_dropout` narrative below predates that bug being found — read it as
> *intent*, not as validated evidence.

Two per-frame feature extractors, sharing the same recurrent head:

- **From-scratch:** 07's `build_cnn_scratch` conv stack (`Conv2D→BatchNorm→MaxPool` ×3), stopped
  at its `GlobalAveragePooling2D → Dense(64, relu)` embedding — its final `Dense(2, softmax)`
  head is dropped, since classification happens after the LSTM here, not per-frame. **No
  per-layer `l2()` regularization here anymore** — see the "Regularization update" note below.
- **MobileNetV2:** frozen `imagenet` weights, `alpha=1.0` and `IMG_SIZE_MOBILENET=128` — a real
  fix attempt for 07's pre-migration 3-class collapse (16.81% accuracy, zero recall on one
  class), not a repeat of its exact `(alpha=0.35, IMG_SIZE=96)` config. `notebook/CLAUDE.md`'s "Cheaper options" section
  diagnoses that collapse as more likely a resolution/`alpha` mismatch (`alpha=0.35` at 96×96
  downsamples to roughly a 3×3 feature map before pooling, discarding most spatial detail in the
  eye/mouth region) than a real verdict against pretraining.

Both are wrapped in `TimeDistributed`, feeding a **single 64-unit `LSTM`** — not the geometric
LSTM's two-layer 128→64 stack. The `TimeDistributed(CNN)` already contributes far more trainable
parameters per timestep (a real conv stack, or MobileNetV2) than the geometric LSTM's 58
hand-engineered floats — this is already the most overfitting-prone model in the project (see the
notebook-level "Calibrated expectation" note above), so the recurrent head is kept smaller here
specifically to offset that, not copied at the same size by default.

The mask is passed explicitly into the `LSTM`'s call (`LSTM(...)(x, mask=mask_input)`) rather
than via a preceding `Masking` layer — see "Building the `tf.data` Pipeline" above for why. The
`TimeDistributed` CNN does still run (wastefully) on zero-padded frames; its output for those
timesteps just doesn't affect the LSTM's hidden-state trajectory, since masked timesteps are
skipped there. Accepted cost of the padding+mask approach.

**Update: geometric feature fusion.** Per timestep, the `TimeDistributed(CNN)` embedding is
concatenated with a normalized 10-feature EAR/MAR/blendshape vector (`GEO_FEATURE_NAMES` above,
extracted per-frame by `09_dataset_creation_cnn_lstm.ipynb` -- see its "Geometric Features for
Fusion" section) before the fused sequence goes into the `LSTM`. Normalization uses a
`tf.keras.layers.Normalization` layer (`geo_normalizer`) adapted once on `df_train`'s real
(non-padded) frames and reused unchanged for both backbones, the same precedent as the class
weights below. `learning_rate` dropped from `0.001` to `1e-4` in an earlier pass (an intermediate
`3e-4` pass showed clear overfitting -- steadily climbing train accuracy against a flat/noisy,
trending-worse `val_macro_f1`); `ReduceLROnPlateau`'s `patience` dropped from `8` to `4` at the
same time, for the same reason.

**Update: regularization, not learning rate, is this round's lever.** Two full training runs at
different learning rates (`3e-4`, then `1e-4` decaying to ~1.25e-5 via `ReduceLROnPlateau`) both
converged to the *same* early `val_macro_f1` peak (epoch ~3-5, ~0.40-0.46) followed by the same
decline shape. Three very different LR magnitudes hitting the same ceiling means the learning
rate isn't the lever holding it down -- so `learning_rate` stays at `1e-4` here, deliberately
**not** lowered further (pushing it lower risks settling into a sharper, more overfit minimum
rather than fixing the ceiling -- LR mainly governs how fast training gets somewhere, not how
well that somewhere generalizes). Two changes that actually target capacity/regularization
instead:
1. **`AdamW`'s decoupled weight decay** (`WEIGHT_DECAY = 1e-4`) replaces plain `Adam` + `l2()`
   scattered across every conv/dense layer — L2-inside-Adam interacts poorly with Adam's
   per-parameter adaptive rates and is a weaker, more erratic form of the same regularization
   AdamW does more cleanly. The `l2()` kernel regularizers above are removed, not kept alongside
   AdamW, to avoid double-regularizing the same weights.
2. **`recurrent_dropout=0.3`** on the `LSTM`, on top of its existing `dropout=0.3` — the LSTM
   previously had zero regularization on its recurrent (hidden-state-to-hidden-state)
   connections, only on its input. This would normally cost the cuDNN fast path, but
   `use_cudnn=False` is already forced here for the pre-padding reason above, so it's free.

**Update: round-2 regularization — all three values raised together, and shared with the
frozen-embedding variant.** The round-1 set above (`WEIGHT_DECAY = 1e-4`, LSTM
`dropout`/`recurrent_dropout = 0.3`, head `Dropout = 0.5`) didn't close the train/val gap on
its own — `notebook/CLAUDE.md`'s "What we found" records the from-scratch backbone still
climbing past 70% train accuracy while `val_macro_f1` peaked early and declined. Round 2
raises all three at once: `WEIGHT_DECAY` `1e-4 → 3e-4`, `LSTM_DROPOUT` / `RECURRENT_DROPOUT`
`0.3 → 0.4`, and `HEAD_DROPOUT` (the `LSTM`'s final `Dropout`, now a named constant)
`0.5 → 0.6`. All three are module-level constants applied unchanged to **both**
`build_cnn_lstm` (from-scratch) and `build_frozen_embedding_lstm` (frozen embedding) below —
same precedent as `geo_normalizer` and the class weights: only the backbone differs between
the two runs, not the regularization strength. **Watch for overshoot into underfitting on
the next run** — if train accuracy also stays low/flat (not just val), these went too far and
the fix is backing them off, not pushing higher again.


**Update: `CosineDecayRestarts` (SGDR) replaces `ReduceLROnPlateau` for the from-scratch model.**
The most recent run added one more, direct piece of evidence against a monotonically-shrinking
LR: as `ReduceLROnPlateau` cut `1e-4` down to `6.25e-6` over that run, `val_macro_f1` at that
lowest LR (~0.22-0.23) was worse than at every higher LR earlier in the *same* run -- not just a
cross-run pattern anymore. `CosineDecayRestarts` takes the same underlying idea one step further:
instead of only ever shrinking the LR, it decays smoothly within each cycle and then jumps the LR
back **up** at each restart, deliberately re-perturbing the model out of whatever region it
settled into rather than letting it settle further. Configured with a ~5-epoch first cycle
(matching where `val_macro_f1` has peaked in every run so far), `t_mul=2.0` (cycles double in
length), `m_mul=0.9` (each restart's peak LR cools slightly), and `alpha=0.05` (each cycle's
trough floors at 5% of that cycle's peak, not near-zero). `ReduceLROnPlateau` is removed for this
model rather than combined with the schedule -- the two don't compose (`ReduceLROnPlateau`
expects a plain scalar `optimizer.lr`, not a `LearningRateSchedule`) and would otherwise fight
each other.


In [ ]:
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Input, Rescaling, Conv2D, BatchNormalization, MaxPooling2D,
    GlobalAveragePooling2D, Dense, Dropout, TimeDistributed, LSTM, Concatenate, Normalization,
)
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.optimizers.schedules import CosineDecayRestarts
import math

EMBED_DIM = 64  # Per-frame embedding size out of either backbone, before the LSTM.

# --- Regularization update: AdamW's decoupled weight decay, not Adam + l2(). ---
# Previously every conv/dense layer below carried kernel_regularizer=l2(0.001) on top of plain
# Adam. Inside Adam, an L2 penalty gets folded into the gradient before Adam's per-parameter
# adaptive moment estimates are computed, which interacts poorly with those adaptive rates and
# can produce less effective, more erratic regularization than true decoupled weight decay (see
# Loshchilov & Hutter's AdamW paper, and https://www.fast.ai/posts/2018-07-02-adam-weight-decay.html).
# Switching to AdamW's `weight_decay` argument gets a cleaner, better-supported version of the
# same regularization -- so the l2() kernel_regularizer args below are REMOVED, not kept
# alongside AdamW: stacking both double-regularizes the same weights and reintroduces the exact
# interaction problem AdamW is meant to fix.
#
# Value: 1e-4, matching 11_cnn_lstm_training_upload.ipynb. A round-2 experiment briefly raised
# this to 3e-4 (with LSTM dropout 0.4 / head 0.6) in the drive-pull variant, but that variant
# was also running with a broken learning rate (see the schedule below), so its "still
# overfitting" evidence was never real -- the model wasn't training at all. Reverted to the
# round-1 set here alongside the LR fix; re-raise only if a genuinely-training run overfits.
WEIGHT_DECAY = 1e-4


def build_scratch_feature_extractor(img_size):
    """Per-frame CNN feature extractor -- 07's build_cnn_scratch conv stack, stopped at its
    GlobalAveragePooling2D -> Dense(64) embedding (its final Dense(3, softmax) head is dropped:
    classification happens after the LSTM here, not per-frame). No per-layer l2() here anymore --
    see the "Regularization update" note above; AdamW's weight_decay (passed at compile time in
    build_cnn_lstm below) covers this instead.
    """
    return Sequential([
        Input(shape=(img_size, img_size, 3)),
        Rescaling(1. / 255),
        Conv2D(16, 3, padding='same', activation='relu'),
        BatchNormalization(),
        MaxPooling2D(),
        Conv2D(32, 3, padding='same', activation='relu'),
        BatchNormalization(),
        MaxPooling2D(),
        Conv2D(64, 3, padding='same', activation='relu'),
        BatchNormalization(),
        MaxPooling2D(),
        GlobalAveragePooling2D(),
        Dense(EMBED_DIM, activation='relu'),
    ], name='scratch_feature_extractor')


def build_mobilenet_feature_extractor(img_size, alpha=1.0, trainable_backbone=False):
    """Per-frame MobileNetV2 feature extractor. Deliberately alpha=1.0 (not 07's 0.35) and a
    larger img_size (see the Model markdown cell above) -- a real attempt at fixing 07's
    resolution/alpha-diagnosed collapse, not a repeat of the exact config that collapsed there.
    No l2() on the embedding Dense here either -- see the "Regularization update" note above.
    """
    base = MobileNetV2(input_shape=(img_size, img_size, 3), alpha=alpha, include_top=False,
                        weights='imagenet')
    base.trainable = trainable_backbone
    inputs = Input(shape=(img_size, img_size, 3))
    x = preprocess_input(inputs)
    x = base(x, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dense(EMBED_DIM, activation='relu')(x)
    model = Model(inputs, x, name='mobilenet_feature_extractor')
    return model, base


# --- Geometric feature normalizer, adapted once on the real (non-padded) training frames -- see
# the "Geometric feature fusion" note in the Model markdown cell above. Reused unchanged for both
# backbones below, same precedent as the class weights: only the backbone differs between the two
# training runs, not how the geometric side of the input is normalized. ---
_geo_train_real_frames = np.array([
    [float(v) for v in frame.split(',')]
    for seq in df_train['geometric_feature_seq']
    for frame in seq.split(';')
], dtype=np.float32)
geo_normalizer = Normalization(axis=-1, name='geo_normalizer')
geo_normalizer.adapt(_geo_train_real_frames)
print(f"Geo feature normalizer adapted on {_geo_train_real_frames.shape[0]} real frames "
      f"({NUM_GEO_FEATURES} features).")


# --- LSTM regularization (round-2 values -- all raised together this round). ---
# dropout only zeroes the LSTM's *input* at each timestep; recurrent_dropout also zeroes its
# hidden-state-to-hidden-state connections, which is where an RNN's own overfitting capacity
# mostly lives (the LSTM had zero recurrent regularization before round 1). recurrent_dropout>0
# normally disables Keras' cuDNN fast path -- a real cost -- but use_cudnn=False is already
# forced below for the pre-padding reason, so it's free here. HEAD_DROPOUT is the LSTM's final
# Dropout before the classification head, promoted to a named constant so the frozen-embedding
# variant below can share the exact same value. Round 1 -> round 2: 0.3 -> 0.4 (both LSTM
# dropouts), 0.5 -> 0.6 (head). All three constants are reused unchanged by
# build_frozen_embedding_lstm, same precedent as geo_normalizer and the class weights.
# Values: 0.3 / 0.3 / 0.5, matching 11_cnn_lstm_training_upload.ipynb (see the WEIGHT_DECAY note
# above for why the drive-pull "round 2" bump to 0.4 / 0.4 / 0.6 was reverted).
LSTM_DROPOUT = 0.3
RECURRENT_DROPOUT = 0.3
HEAD_DROPOUT = 0.5


def build_cnn_lstm(feature_extractor, img_size, max_timesteps=MAX_TIMESTEPS_IMG,
                    num_geo_features=NUM_GEO_FEATURES, num_classes=num_classes,
                    learning_rate=1e-4, name='cnn_lstm'):
    """Wraps a per-frame feature extractor in TimeDistributed, concatenates its output sequence
    with the normalized per-frame geometric feature sequence (EAR/MAR/blendshapes), feeds the
    fused sequence into a single LSTM(64) with an explicit mask, then a small classification
    head. See the Model markdown cell above for why a single, modestly-sized LSTM layer rather
    than the geometric LSTM's two-layer stack.

    `learning_rate` defaults to a plain 1e-4 float here, but the from-scratch call below passes a
    `CosineDecayRestarts` schedule object instead (Keras optimizers accept either) -- see the
    schedule definition below for why. Four full training runs now (learning_rate=3e-4, then 1e-4
    decaying via ReduceLROnPlateau down to ~1.25e-5, then again down to 6.25e-6) all converged to
    the SAME early val_macro_f1 peak (epoch ~3-5, ~0.40-0.46) followed by the same decline shape,
    and within the most recent run val_macro_f1 at the lowest LR (6.25e-6) was directly worse than
    at every higher LR earlier in that same run -- a monotonically-shrinking LR was never the
    lever that raises this ceiling, and now has direct within-run evidence of making things worse.
    AdamW's weight_decay (this function's compile step) and recurrent_dropout (below) target that
    ceiling more directly; the schedule below is a further, different experiment on top.
    """
    image_input = Input(shape=(max_timesteps, img_size, img_size, 3), name='images')
    geo_input = Input(shape=(max_timesteps, num_geo_features), name='geo_features')
    mask_input = Input(shape=(max_timesteps,), dtype=tf.bool, name='mask')

    cnn_embeddings = TimeDistributed(feature_extractor)(image_input)  # (T, EMBED_DIM)
    normalized_geo = geo_normalizer(geo_input)                        # (T, num_geo_features)
    fused = Concatenate(axis=-1)([cnn_embeddings, normalized_geo])    # (T, EMBED_DIM + num_geo_features)

    # Fix: Set use_cudnn=False because sequences are pre-padded, not right-padded
    x = LSTM(64, dropout=LSTM_DROPOUT, recurrent_dropout=RECURRENT_DROPOUT, use_cudnn=False)(fused, mask=mask_input)
    x = Dropout(HEAD_DROPOUT)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model([image_input, geo_input, mask_input], outputs, name=name)
    model.compile(optimizer=AdamW(learning_rate=learning_rate, weight_decay=WEIGHT_DECAY),
                  loss=build_loss(), metrics=['accuracy'])
    return model


# --- Learning rate schedule: CosineDecayRestarts (SGDR) instead of a flat LR handed to
# ReduceLROnPlateau. Every run so far shows the same shape -- val_macro_f1 peaks early (epoch
# ~3-5) then declines as training continues -- and ReduceLROnPlateau's own monotonic decay has
# now directly made that decline WORSE, not better (see the docstring above and
# notebook/CLAUDE.md's "What we found"). CosineDecayRestarts implements the same underlying idea
# behind that observation -- let the LR go back UP periodically to knock training out of whatever
# region it settled into, rather than only ever shrinking it -- in its established, well-tested
# form, rather than a custom callback reacting to val_macro_f1 directly. Deliberately NOT
# metric-reactive: val_macro_f1 comes from only 234 validation windows (6 subjects) and is noisy
# enough that a scheme reacting to it in real time risks mistaking noise for "stuck".
STEPS_PER_EPOCH_SCRATCH = math.ceil(len(df_train) / BATCH_SIZE)
lr_schedule_scratch = CosineDecayRestarts(
    initial_learning_rate=1e-4,   # PEAK LR of the schedule (it decays from here and restarts
                                   # back toward here). A prior drive-pull edit left this at
                                   # 1e-8, which trained the from-scratch model at ~1e-8
                                   # throughout -- train accuracy sat flat at ~0.50 and nothing
                                   # learned. Matches 11_cnn_lstm_training_upload.ipynb.
    first_decay_steps=STEPS_PER_EPOCH_SCRATCH * 5,  # ~5 epochs per first cycle -- matches where
                                                      # val_macro_f1 peaked in every run so far.
    t_mul=2.0,   # each subsequent cycle is 2x longer than the last (~5 -> 10 -> 20 epochs, ...).
    m_mul=0.9,   # each restart's peak LR is 90% of the previous cycle's -- a mild cooldown across
                 # restarts, not a full reset to 1e-4 every time.
    alpha=0.05,  # floors each cycle's trough at 5% of that cycle's peak, not near-zero -- avoids
                 # the "very low LR made things worse" failure mode already measured directly.
)

model_scratch = build_cnn_lstm(build_scratch_feature_extractor(IMG_SIZE), img_size=IMG_SIZE,
                                learning_rate=lr_schedule_scratch, name='cnn_lstm_scratch')
model_scratch.summary()


## CNN+LSTM Training (From-Scratch Backbone)


> **Correction.** The prior drive-pull run of this cell trained at a broken learning rate
> (~`1e-8`, see the Model Definition correction above) and never actually learned — its
> `val_macro_f1` log and any "class weighting didn't help" reading from it are not real. With
> the LR fixed, this run also introduces **balanced resampling** (see the resampling cell): the
> training set is drawn 50/50 from the two classes so every `BATCH_SIZE=8` batch has a stable
> mix, and `class_weight` is set to `None` — the `DROWSY_WEIGHT_BOOST` / `NOT_DROWSY_WEIGHT_DAMPEN`
> knobs below only apply on the `USE_CLASS_WEIGHTS=True` / `USE_BALANCED_SAMPLING=False` path.
> The final `Drowsy` operating point is set afterward by the decision-threshold cell, not by
> the loss weighting.

`class_weight`, computed from the training set's actual **window-level** label distribution
(pooling all 4 durations together -- same precedent as `03_model_training_lstm.ipynb` pooling its
own multi-duration window mix into one training set). **Update: disabled for this run
(`USE_CLASS_WEIGHTS = False`).** Earlier passes applied class weighting on top of `'balanced'`
(with a `Drowsy` boost; a 3-class version also dampened the ambiguous middle class, which no
longer exists) and all showed the same early-peak-then-decline `val_macro_f1` pattern regardless
of learning rate -- since that pattern persisted across very different learning rates, the
learning rate was ruled out as the cause (see the Model Definition markdown above); the
class-weight skew combined with a small `BATCH_SIZE=8` is the next candidate (individual batches
dominated by one heavily-weighted example, plausibly driving epoch-to-epoch `val_macro_f1`
noise). This run isolates that by training with `class_weight=None` -- a genuinely uniform,
unweighted loss, not just the `Drowsy` boost turned off on top of `'balanced'`. **Real tradeoff, not hidden:** `Drowsy` is the single most
safety-critical class in this project -- watch its recall in the eval cells below; if it craters
here, that's a real cost of this experiment to weigh against any accuracy/macro-F1 gain. **Worth
noting either way: the late-run decline never actually reached the saved model** -- `ModelCheckpoint`
below keeps only the best `val_macro_f1` epoch (`save_best_only=True`), so it's wasted training
time, not a corrupted result. `MacroF1Callback` and the val-macro-F1-based `EarlyStopping`/
`ModelCheckpoint` setup are ported from 07 -- see that notebook for why raw `val_accuracy` isn't
a safe selection metric under class imbalance. **`ReduceLROnPlateau` is no longer part of this
model's callbacks** -- `learning_rate` is now a `CosineDecayRestarts` schedule handed directly to
`AdamW` (see the Model Definition markdown above), which handles decay/restarts on its own and
doesn't compose with a plateau-based callback.


In [ ]:
from tensorflow.keras.callbacks import Callback, EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import os


class MacroF1Callback(Callback):
    """Adds `val_macro_f1` to the epoch's `logs` dict so EarlyStopping/ModelCheckpoint below can
    select on it instead of raw accuracy -- ported from 07, works unmodified against this
    notebook's ((images, mask), label) dataset shape: only the top-level (inputs, label)
    structure matters for `for _, y in self.val_ds` and `self.model.predict(self.val_ds)`.
    """
    def __init__(self, val_ds):
        super().__init__()
        self.val_ds = val_ds

    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}
        y_true = np.concatenate([y.numpy() for _, y in self.val_ds])
        y_pred = np.argmax(self.model.predict(self.val_ds, verbose=0), axis=1)
        logs['val_macro_f1'] = f1_score(y_true, y_pred, average='macro',
                                         labels=list(range(num_classes)), zero_division=0)


# --- Class balancing ---
# Primary path is USE_BALANCED_SAMPLING (set in the tf.data cell): the training set is drawn
# 50/50 from the two classes, so every BATCH_SIZE=8 batch has a stable mix and no `class_weight`
# is needed. The `class_weight` path below is the fallback for USE_BALANCED_SAMPLING=False --
# kept because at this batch size a heavily-reweighted example can dominate a batch (a suspected
# source of epoch-to-epoch val_macro_f1 noise), which is exactly why sampling is preferred.
# Either way, the FINAL Drowsy operating point is set by the decision-threshold cell after
# evaluation (safety-first: hit a Drowsy recall floor, then best precision), not by the loss.
DROWSY_LABEL = CLASS_NAMES.index('Drowsy')          # 1
NOT_DROWSY_LABEL = CLASS_NAMES.index('Not Drowsy')  # 0
USE_CLASS_WEIGHTS = True         # only consulted when USE_BALANCED_SAMPLING is False
DROWSY_WEIGHT_BOOST = 1.5        # boost on top of sklearn 'balanced' (fallback path only)
NOT_DROWSY_WEIGHT_DAMPEN = 0.6   # extra dampen on Not Drowsy's balanced weight (fallback path only)

if USE_BALANCED_SAMPLING:
    weight_dict = None
    print("USE_BALANCED_SAMPLING=True -> class_weight=None; the 50/50 tf.data sampler balances "
          "every batch. DROWSY_WEIGHT_BOOST / NOT_DROWSY_WEIGHT_DAMPEN are inert this run.")
elif USE_CLASS_WEIGHTS:
    train_labels = (df_train['level'].to_numpy(dtype=np.int32) - 1)
    weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
    weight_dict = dict(zip(np.unique(train_labels).tolist(), weights))
    weight_dict[DROWSY_LABEL] *= DROWSY_WEIGHT_BOOST
    weight_dict[NOT_DROWSY_LABEL] *= NOT_DROWSY_WEIGHT_DAMPEN
    print(f"Class weights (Drowsy x{DROWSY_WEIGHT_BOOST}, Not Drowsy dampened "
          f"x{NOT_DROWSY_WEIGHT_DAMPEN}): {weight_dict}")
else:
    weight_dict = None
    print("USE_BALANCED_SAMPLING=False, USE_CLASS_WEIGHTS=False -> uniform unweighted loss.")

macro_f1_cb = MacroF1Callback(val_ds)
early_stopping = EarlyStopping(monitor='val_macro_f1', mode='max', patience=15,
                                restore_best_weights=True)
checkpoint_scratch = ModelCheckpoint(
    filepath=os.path.join(models_folder, 'best_cnn_lstm_scratch.keras'),
    monitor='val_macro_f1', mode='max', save_best_only=True,
)
# No ReduceLROnPlateau here anymore -- the from-scratch model's learning_rate is now a
# CosineDecayRestarts schedule (see cell above), which already handles LR decay/restarts on its
# own schedule. Combining a LearningRateSchedule object with ReduceLROnPlateau doesn't compose
# cleanly (ReduceLROnPlateau expects to read/write a plain scalar optimizer.lr) and would fight
# the schedule's own restarts, so it's removed rather than kept alongside it.

print("🚀 Training CNN+LSTM (from-scratch backbone)...")
history_scratch = model_scratch.fit(
    train_ds,
    epochs=100,
    steps_per_epoch=STEPS_PER_EPOCH_SCRATCH if USE_BALANCED_SAMPLING else None,  # train_ds is infinite when balanced
    validation_data=val_ds,
    callbacks=[macro_f1_cb, early_stopping, checkpoint_scratch],
    class_weight=weight_dict,
    verbose=1,
)


## Evaluation (From-Scratch Backbone)

Same shape as 07: `classification_report`/confusion matrix/macro-F1 on the held-out test
subjects, loaded from the best `val_macro_f1` checkpoint. Also reports accuracy **broken down by
`window_duration_sec`** — something 07 has no equivalent of, since it has no duration dimension —
which directly answers whether more temporal context actually helps, the whole premise of
building this model. `test_ds` is never shuffled, so `y_pred`'s order lines up 1:1 with
`df_test`'s row order.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import os
import numpy as np
from tensorflow.keras.optimizers import Adam

def evaluate_variant(model_path, fallback_model, test_dataset, test_df, variant_label):
    """Loads the best checkpoint (falling back to the in-memory model if the checkpoint file
    isn't found), evaluates against test_dataset, prints a classification report + confusion
    matrix, and returns (y_true, y_pred, per_duration_accuracy) for later comparison.
    """
    custom_objects = {'SparseCategoricalFocalLoss': SparseCategoricalFocalLoss}
    if os.path.exists(model_path):
        print(f"Loading best {variant_label} checkpoint from {model_path}...")
        # Note: compile=False avoids certain loading errors; we re-compile or just use for inference.
        model_eval = tf.keras.models.load_model(model_path, custom_objects=custom_objects, compile=False)

        # Fix for cuDNN error: Ensure LSTM layers use_cudnn=False for pre-padded sequences
        for layer in model_eval.layers:
            if isinstance(layer, tf.keras.layers.LSTM):
                layer.use_cudnn = False

        # RE-COMPILE: Necessary because we loaded with compile=False
        model_eval.compile(optimizer=Adam(learning_rate=1e-4), loss=build_loss(), metrics=['accuracy'])
    else:
        print(f"Best {variant_label} checkpoint not found, falling back to in-memory model.")
        model_eval = fallback_model

    loss, accuracy = model_eval.evaluate(test_dataset, verbose=0)
    y_true = np.concatenate([y.numpy() for _, y in test_dataset])
    y_pred = np.argmax(model_eval.predict(test_dataset), axis=1)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

    print(f"\n{variant_label} Test Loss: {loss:.4f}")
    print(f"{variant_label} Test Accuracy: {accuracy:.4f}")
    print(f"{variant_label} Test Macro-F1: {macro_f1:.4f}")
    print(f"\nClassification Report ({variant_label}):")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

    conf_matrix = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(f'Confusion Matrix ({variant_label})')
    plt.show()

    # --- Per-duration accuracy breakdown -- the question this whole model exists to answer ---
    eval_df = test_df.reset_index(drop=True).copy()
    eval_df['correct'] = (y_true == y_pred)
    per_duration = eval_df.groupby('window_duration_sec')['correct'].agg(['mean', 'count'])
    per_duration.columns = ['accuracy', 'n_windows']
    print(f"\nAccuracy by window duration ({variant_label}):")
    print(per_duration)

    return y_true, y_pred, per_duration


y_true_scratch, y_pred_scratch, per_duration_scratch = evaluate_variant(
    os.path.join(models_folder, 'best_cnn_lstm_scratch.keras'),
    model_scratch, test_ds, df_test, 'CNN+LSTM (from-scratch)',
)

if 'history_scratch' in globals():
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history_scratch.history['accuracy'], label='Training Accuracy')
    plt.plot(history_scratch.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Model Accuracy'); plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history_scratch.history['loss'], label='Training Loss')
    plt.plot(history_scratch.history['val_loss'], label='Validation Loss')
    plt.title('Model Loss'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend()
    plt.tight_layout()
    plt.show()

## Operating-Point Selection (decision threshold on p(Drowsy))

Every evaluation above reports metrics at the implicit **argmax** threshold — 0.5 on a 2-class
softmax. That is almost never the right operating point for a safety-critical minority class,
and it is the single most direct "make the model prefer `Drowsy`" lever we have that costs no
retraining.

This tooling sweeps the threshold on `p(Drowsy)` using the **validation set only**, picks an
operating point, then reports **test-set** metrics at that threshold next to the argmax
baseline. Selection is *safety-first* (project decision): among thresholds whose validation
`Drowsy` recall ≥ `DROWSY_RECALL_FLOOR`, take the one with the best `Drowsy` precision (that is
the highest such threshold — recall falls and precision rises as the threshold rises). If no
threshold clears the floor, fall back to the max-recall threshold. The max-macro-F1 threshold
is also printed so the precision given up to hit the recall floor is explicit.

It also fits an **isotonic calibrator** on the validation `p(Drowsy)` and re-runs the sweep on
calibrated probabilities, as a check on whether the chosen operating point transfers from
validation to test more reliably once the probabilities are calibrated.

The chosen raw-probability threshold is written to `<checkpoint>.threshold.json` so
`src/cv-argus` can apply the same operating point at inference instead of `argmax`.


In [ ]:
# --- Shared operating-point tooling (used by the from-scratch and frozen-embedding cells below) ---
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (classification_report, confusion_matrix, f1_score,
                             precision_score, recall_score)
from sklearn.isotonic import IsotonicRegression

DROWSY_IDX = CLASS_NAMES.index('Drowsy')          # 1
DROWSY_RECALL_FLOOR = 0.70                         # min VALIDATION Drowsy recall to accept -- tune me
THRESHOLD_GRID = np.round(np.arange(0.05, 0.951, 0.01), 4)


def _load_for_eval(model_path, fallback_model):
    """Same load pattern as evaluate_variant: best checkpoint if present, else the in-memory
    model; force LSTM.use_cudnn=False for the pre-padded sequences; no recompile needed (we only
    call .predict)."""
    import os
    if os.path.exists(model_path):
        m = tf.keras.models.load_model(
            model_path, custom_objects={'SparseCategoricalFocalLoss': SparseCategoricalFocalLoss},
            compile=False)
        for layer in m.layers:
            if isinstance(layer, tf.keras.layers.LSTM):
                layer.use_cudnn = False
        return m
    print(f"  (checkpoint {model_path} missing -- using in-memory model)")
    return fallback_model


def _p_drowsy(model, ds):
    y_true = np.concatenate([y.numpy() for _, y in ds])
    proba = model.predict(ds, verbose=0)
    return y_true.astype(int), proba[:, DROWSY_IDX]


def _sweep(y_true, p_drowsy, grid=THRESHOLD_GRID):
    rows = []
    for t in grid:
        y_hat = (p_drowsy >= t).astype(int)   # binary, Drowsy == 1
        rows.append(dict(
            t=float(t),
            macro_f1=f1_score(y_true, y_hat, labels=[0, 1], average='macro', zero_division=0),
            drowsy_p=precision_score(y_true, y_hat, pos_label=1, zero_division=0),
            drowsy_r=recall_score(y_true, y_hat, pos_label=1, zero_division=0),
        ))
    return rows


def _pick(rows, floor):
    clear = [r for r in rows if r['drowsy_r'] >= floor]
    if clear:
        star = max(clear, key=lambda r: (r['drowsy_p'], r['t']))
        note = f"highest t with val Drowsy recall >= {floor:.2f} (best precision among those)"
    else:
        star = max(rows, key=lambda r: r['drowsy_r'])
        note = f"NO threshold reached val Drowsy recall {floor:.2f} -- fell back to max-recall t"
    f1_star = max(rows, key=lambda r: r['macro_f1'])
    return star, f1_star, note


def _report_at(y_true, p_drowsy, t, header):
    y_hat = (p_drowsy >= t).astype(int)
    mf1 = f1_score(y_true, y_hat, labels=[0, 1], average='macro', zero_division=0)
    print(f"\n--- {header} (threshold = {t:.2f}) ---")
    print(f"accuracy {np.mean(y_true == y_hat):.4f} | macro-F1 {mf1:.4f}")
    print(classification_report(y_true, y_hat, target_names=CLASS_NAMES, zero_division=0))
    print("confusion (rows=true, cols=pred):\n", confusion_matrix(y_true, y_hat, labels=[0, 1]))
    return dict(threshold=float(t), accuracy=float(np.mean(y_true == y_hat)), macro_f1=float(mf1),
               drowsy_p=float(precision_score(y_true, y_hat, pos_label=1, zero_division=0)),
               drowsy_r=float(recall_score(y_true, y_hat, pos_label=1, zero_division=0)))


def operating_point_analysis(variant_label, checkpoint_path, fallback_model,
                             val_dataset, test_dataset, test_df=None):
    """Full pipeline for one model variant. Returns the chosen raw-probability threshold and
    also writes it to <checkpoint_path>.threshold.json."""
    print("=" * 78)
    print(f"OPERATING-POINT ANALYSIS -- {variant_label}")
    print("=" * 78)
    model = _load_for_eval(checkpoint_path, fallback_model)
    yv, pv = _p_drowsy(model, val_dataset)
    yt, pt = _p_drowsy(model, test_dataset)
    print(f"val: {len(yv)} windows ({int(yv.sum())} Drowsy) | test: {len(yt)} windows "
          f"({int(yt.sum())} Drowsy)")

    # --- raw probabilities ---
    rows = _sweep(yv, pv)
    star, f1_star, note = _pick(rows, DROWSY_RECALL_FLOOR)
    print(f"\nchosen t* (val)        = {star['t']:.2f}  [{note}]")
    print(f"  val   @ t*: macro-F1 {star['macro_f1']:.4f}  Drowsy P {star['drowsy_p']:.3f}  R {star['drowsy_r']:.3f}")
    print(f"max-macro-F1 t (val)   = {f1_star['t']:.2f}  "
          f"(macro-F1 {f1_star['macro_f1']:.4f}, Drowsy P {f1_star['drowsy_p']:.3f} R {f1_star['drowsy_r']:.3f})")

    base = _report_at(yt, pt, 0.50, f"{variant_label} TEST @ argmax baseline")
    tuned = _report_at(yt, pt, star['t'], f"{variant_label} TEST @ chosen t*")

    # --- isotonic-calibrated probabilities (diagnostic) ---
    iso = IsotonicRegression(out_of_bounds='clip', y_min=0.0, y_max=1.0).fit(pv, yv)
    pv_c, pt_c = iso.predict(pv), iso.predict(pt)
    rows_c = _sweep(yv, pv_c)
    star_c, _, note_c = _pick(rows_c, DROWSY_RECALL_FLOOR)
    cal = _report_at(yt, pt_c, star_c['t'], f"{variant_label} TEST @ calibrated t* ({star_c['t']:.2f})")
    print(f"\ncalibration check: val macro-F1 @ chosen "
          f"{max(r['macro_f1'] for r in rows_c):.4f} vs test {cal['macro_f1']:.4f} "
          f"(gap {abs(max(r['macro_f1'] for r in rows_c) - cal['macro_f1']):.4f}; "
          f"smaller = operating point transfers)")

    # --- threshold-vs-metric curve ---
    ts = [r['t'] for r in rows]
    plt.figure(figsize=(7, 4))
    plt.plot(ts, [r['drowsy_r'] for r in rows], label='val Drowsy recall')
    plt.plot(ts, [r['drowsy_p'] for r in rows], label='val Drowsy precision')
    plt.plot(ts, [r['macro_f1'] for r in rows], label='val macro-F1')
    plt.axhline(DROWSY_RECALL_FLOOR, ls=':', c='grey'); plt.axvline(star['t'], ls='--', c='k')
    plt.xlabel('threshold on p(Drowsy)'); plt.legend(); plt.title(f'{variant_label} -- val threshold sweep')
    plt.tight_layout(); plt.show()

    # --- per-duration at t* (11 only) ---
    if test_df is not None:
        d = test_df.reset_index(drop=True).copy()
        d['correct'] = (yt == (pt >= star['t']).astype(int))
        print(f"\nTest accuracy by window duration @ t* ({variant_label}):")
        print(d.groupby('window_duration_sec')['correct'].agg(['mean', 'count']))

    # --- persist raw-probability threshold for src/cv-argus ---
    out = f"{checkpoint_path}.threshold.json"
    with open(out, 'w') as fh:
        json.dump(dict(variant=variant_label, drowsy_index=DROWSY_IDX,
                       drowsy_recall_floor=DROWSY_RECALL_FLOOR,
                       threshold=star['t'], selection=note,
                       test_at_threshold=tuned, test_at_argmax=base), fh, indent=2)
    print(f"\nwrote {out}")
    return star['t']


In [ ]:
# Operating point for the from-scratch CNN+LSTM. df_test is passed so the per-window_duration_sec
# breakdown is computed at the chosen threshold too.
t_star_scratch = operating_point_analysis(
    'CNN+LSTM (from-scratch)',
    os.path.join(models_folder, 'best_cnn_lstm_scratch.keras'),
    model_scratch, val_ds, test_ds, test_df=df_test,
)


## Frozen-CNN-Embedding + LSTM Variant (Cheaper Alternative)

Added alongside the from-scratch and MobileNetV2 variants above, not replacing either -- see
`notebook/CLAUDE.md`'s "Cheaper options considered alongside the full CNN+LSTM build" (option 3)
and "What we found": the from-scratch CNN+LSTM's pre-migration 3-class test result (32.22% accuracy,
0.3251 macro-F1 -- at the 33.3% random-guess chance for that balanced 3-class test set) was a
real train/test generalization-gap problem that survived every regularization change tried on it
(`AdamW` weight decay, LSTM dropout/recurrent dropout, focal loss, class-weight toggling,
`CosineDecayRestarts`). No binary run has happened yet -- but that consistency points at model
capacity relative to subject count, not an undiscovered hyperparameter, and collapsing to two
classes raises the random-guess floor to 50% rather than closing that gap -- so this section
tries a genuinely lower-capacity model instead of tuning the same one further.

Instead of training a CNN backbone end-to-end inside the recurrent graph
(`TimeDistributed(CNN) -> LSTM`, what the from-scratch/MobileNetV2 variants above both do), this
reuses `07_cnn_training.ipynb`'s already-trained single-frame CNN as a **frozen** per-frame
feature extractor: its 64-dim penultimate-layer embedding (`GlobalAveragePooling2D -> Dense(64)`,
before its classification head) is computed once per crop and concatenated with the same 10-dim
`geometric_feature_seq` (EAR/MAR/blendshapes) already used above, then only a small `LSTM(64)` +
classification head is trained on top of that fused, precomputed sequence. Far fewer trainable
parameters than the from-scratch variant -- no CNN weights are updated at all here -- which is
the whole point: a model with less capacity to memorize 48 subjects' worth of training windows by
identity rather than by drowsiness cues.

**Real tradeoff, stated up front:** because the CNN side is frozen and its embeddings are
precomputed once rather than recomputed per batch, this variant does **not** get the image-space
augmentation (flip/brightness/rotation/zoom/contrast/noise) the from-scratch pipeline applies
every epoch -- augmenting would mean re-running the CNN per augmented copy, which defeats the
"cheap" premise here. If this variant still overfits, that's the first thing to add back (e.g.
caching a handful of augmented embedding copies per crop), not a sign the frozen-embedding
approach itself doesn't work.

### Loading `07`'s Frozen CNN Checkpoint

This variant reuses `07_cnn_training.ipynb`'s already-trained single-frame CNN as a frozen
feature extractor. Drive is mounted, so this cell checks `models_folder` on Drive first and only
prompts for a manual upload if `best_cnn_scratch_face_crops.keras` is genuinely missing there.
Skip this (and the rest of this section) if you only want the from-scratch backbone above.

In [ ]:
from google.colab import files

_checkpoint_target = os.path.join(models_folder, 'best_cnn_scratch_face_crops.keras')
if not os.path.exists(_checkpoint_target):
    print("Upload best_cnn_scratch_face_crops.keras (07_cnn_training.ipynb's trained checkpoint):")
    _uploaded_ckpt = files.upload()
    if not _uploaded_ckpt:
        raise RuntimeError("No file uploaded -- re-run this cell and select 07's .keras checkpoint.")
    _ckpt_filename = next(iter(_uploaded_ckpt))
    os.replace(_ckpt_filename, _checkpoint_target)
    print(f"Saved to {_checkpoint_target}.")
else:
    print(f"{_checkpoint_target} already present, skipping upload.")


In [ ]:
import tensorflow as tf
import numpy as np
import os

# --- Load 07's already-trained single-frame CNN as a frozen feature extractor ---
cnn_scratch_checkpoint_path = os.path.join(models_folder, 'best_cnn_scratch_face_crops.keras')
if not os.path.exists(cnn_scratch_checkpoint_path):
    raise FileNotFoundError(
        f"'{cnn_scratch_checkpoint_path}' not found. Run 07_cnn_training.ipynb first -- this "
        "variant reuses its trained CNN as a frozen feature extractor rather than training a new "
        "one from scratch."
    )

_cnn_07 = tf.keras.models.load_model(
    cnn_scratch_checkpoint_path,
    custom_objects={'SparseCategoricalFocalLoss': SparseCategoricalFocalLoss},
    compile=False,
)
_cnn_07.trainable = False  # Frozen backbone

# Force initialization by calling the model with dummy data.
_ = _cnn_07(tf.zeros((1, IMG_SIZE, IMG_SIZE, 3)))

# The 64-dim embedding is 07's penultimate layer: GlobalAveragePooling2D -> Dense(64, relu)
_dense_layers = [l for l in _cnn_07.layers if isinstance(l, tf.keras.layers.Dense)]
if len(_dense_layers) < 2:
    raise ValueError(
        f"Expected at least 2 Dense layers in the loaded CNN, found {len(_dense_layers)}."
    )
_embedding_layer = _dense_layers[-2]
EMBED_DIM_FROZEN = _embedding_layer.output.shape[-1]

# FIX: Instead of _cnn_07.input, we use the input of the first layer (InputLayer)
# to avoid the 'never been called' error when building a sub-model from a loaded file.
model_input = _cnn_07.layers[0].input

frozen_cnn_embedder = tf.keras.Model(
    inputs=model_input,
    outputs=_embedding_layer.output,
    name='frozen_cnn_embedder',
)
print(f"Loaded frozen CNN embedder from {cnn_scratch_checkpoint_path} "
      f"(embedding layer: '{_embedding_layer.name}', EMBED_DIM_FROZEN={EMBED_DIM_FROZEN}).")

In [ ]:
# --- Precompute a frozen embedding for every unique crop image referenced by any window ---
# Crops are shared across many (often overlapping/tiled) windows of different durations -- 09's
# window builder tiles non-overlapping windows per duration config from the same underlying
# frames -- so embedding each *unique* image once and caching it is both correct and much
# cheaper than recomputing per window. Uses ALL windows in df_cnn_lstm_windows (not just
# df_train), since df_val/df_test crops need cached embeddings too.
_all_image_paths = pd.unique(df_cnn_lstm_windows['image_paths'].str.split(';').explode())
print(f"Precomputing frozen CNN embeddings for {len(_all_image_paths)} unique crop images...")

def _load_for_embedding(path):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_jpeg(image_bytes, channels=3)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])  # Same IMG_SIZE=96 as 07 -- same crops.
    return image

_embed_ds = tf.data.Dataset.from_tensor_slices(_all_image_paths)
_embed_ds = _embed_ds.map(_load_for_embedding, num_parallel_calls=tf.data.AUTOTUNE)
_embed_ds = _embed_ds.batch(128).prefetch(tf.data.AUTOTUNE)

_embeddings = frozen_cnn_embedder.predict(_embed_ds, verbose=1)
embedding_by_path = dict(zip(_all_image_paths, _embeddings))
print(f"✅ Cached {len(embedding_by_path)} embeddings, shape per crop: {_embeddings.shape[1:]}")

In [ ]:
def build_fused_arrays(df, embedding_by_path, max_timesteps, embed_dim, num_geo_features):
    """Turns each window row into a padded (max_timesteps, embed_dim + num_geo_features) fused
    feature array plus a boolean mask, entirely in numpy -- no tf.data image pipeline needed here
    since the CNN side is a cached lookup rather than a per-batch forward pass. Same zero-
    pre-pad convention as load_window above (zeros first, real frames last), so results stay
    directly comparable to the from-scratch variant.
    """
    n = len(df)
    fused_dim = embed_dim + num_geo_features
    X = np.zeros((n, max_timesteps, fused_dim), dtype=np.float32)
    mask = np.zeros((n, max_timesteps), dtype=bool)
    y = (df['level'].to_numpy(dtype=np.int32) - 1)  # 0-indexed: 0=Not Drowsy, 1=Drowsy

    for i, row in enumerate(df.itertuples(index=False)):
        paths = row.image_paths.split(';')
        n_real = len(paths)
        embeds = np.stack([embedding_by_path[p] for p in paths])            # (n_real, embed_dim)
        geo_frames = [
            [float(v) for v in frame.split(',')] for frame in row.geometric_feature_seq.split(';')
        ]
        geo = np.array(geo_frames, dtype=np.float32)                        # (n_real, num_geo_features)
        fused = np.concatenate([embeds, geo], axis=-1)                      # (n_real, fused_dim)

        pad_amount = max_timesteps - n_real
        X[i, pad_amount:] = fused
        mask[i, pad_amount:] = True

    return X, mask, y

# Fix: The `embedding_by_path` dictionary keys were generated using original Drive paths,
# but `df_train` (and `df_val`, `df_test`) now use localized paths. Update the keys
# in `embedding_by_path` to match the localized paths.
if list(embedding_by_path.keys())[0].startswith('/content/drive/MyDrive'):
    print("Adjusting embedding_by_path keys to use local file paths...")
    new_embedding_by_path = {
        os.path.join(LOCAL_FACE_CROPS_DIR, os.path.basename(k)): v
        for k, v in embedding_by_path.items()
    }
    embedding_by_path = new_embedding_by_path
    print(f"Keys adjusted. Example new key: {list(embedding_by_path.keys())[0]}")

X_train_fe, mask_train_fe, y_train_fe = build_fused_arrays(
    df_train, embedding_by_path, MAX_TIMESTEPS_IMG, EMBED_DIM_FROZEN, NUM_GEO_FEATURES)
X_val_fe, mask_val_fe, y_val_fe = build_fused_arrays(
    df_val, embedding_by_path, MAX_TIMESTEPS_IMG, EMBED_DIM_FROZEN, NUM_GEO_FEATURES)
X_test_fe, mask_test_fe, y_test_fe = build_fused_arrays(
    df_test, embedding_by_path, MAX_TIMESTEPS_IMG, EMBED_DIM_FROZEN, NUM_GEO_FEATURES)

print(f"Fused feature shapes -- train: {X_train_fe.shape}, val: {X_val_fe.shape}, "
      f"test: {X_test_fe.shape}")

### Model Definition (Frozen CNN Embedding + LSTM)

Reuses the same `LSTM(64)` + `Dropout(HEAD_DROPOUT)` + classification-head shape as
`build_cnn_lstm` above — including its round-2 regularization constants (`WEIGHT_DECAY`,
`LSTM_DROPOUT`, `RECURRENT_DROPOUT`, `HEAD_DROPOUT`) reused unchanged, but with a single fused-feature `Input` instead of separate images/geo/mask inputs feeding
a `TimeDistributed(CNN)` -- there's no CNN left inside this graph to wrap, since it already ran
once during the precompute step above. A `Normalization` layer, adapted on the real (non-padded)
training frames, standardizes the fused vector before the LSTM -- the same role `geo_normalizer`
plays above, just covering the CNN-embedding columns too now rather than only the geometric
ones.

In [ ]:
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense, Normalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import AdamW

FUSED_DIM_FROZEN = EMBED_DIM_FROZEN + NUM_GEO_FEATURES

fused_normalizer_frozen = Normalization(axis=-1, name='fused_normalizer_frozen')
fused_normalizer_frozen.adapt(X_train_fe[mask_train_fe])
print(f"Fused feature normalizer adapted on {mask_train_fe.sum()} real training frames "
      f"({FUSED_DIM_FROZEN} features: {EMBED_DIM_FROZEN} CNN embedding + {NUM_GEO_FEATURES} "
      "geometric).")


def build_frozen_embedding_lstm(max_timesteps=MAX_TIMESTEPS_IMG, fused_dim=FUSED_DIM_FROZEN,
                                 num_classes=num_classes, learning_rate=1e-3,
                                 name='cnn_lstm_frozen_embedding'):
    """Deliberately much smaller than build_cnn_lstm above: with the CNN frozen and precomputed,
    there's no TimeDistributed(CNN) in this graph at all -- only an LSTM(64) + small head train,
    on top of already-fused (embedding + geometric) per-timestep vectors. Far fewer trainable
    parameters than the end-to-end from-scratch CNN+LSTM, by design -- see the markdown cell
    above for why a lower-capacity model is the point of this variant, not an oversight.


    `learning_rate` defaults to 1e-3 here, deliberately higher than build_cnn_lstm's 1e-4: this
    graph is a single small LSTM(64) + head over precomputed features, with no CNN in the loop,
    so it tolerates (and needs) a faster rate. A prior drive-pull edit left this at 1e-8 and
    never overrode it at the call site below, so this model also trained at ~1e-8 -- its
    val_accuracy / val_macro_f1 were frozen at the untrained-init values for the whole run.
    Sweep 1e-3 / 3e-4 on the first genuinely-training run.
    """
    fused_input = Input(shape=(max_timesteps, fused_dim), name='fused_features')
    mask_input = Input(shape=(max_timesteps,), dtype=tf.bool, name='mask')

    x = fused_normalizer_frozen(fused_input)
    # Shares LSTM_DROPOUT / RECURRENT_DROPOUT / HEAD_DROPOUT (and WEIGHT_DECAY) with
    # build_cnn_lstm above -- only the backbone differs between the two variants, not the
    # regularization strength, same precedent as geo_normalizer and the class weights.
    x = LSTM(64, dropout=LSTM_DROPOUT, recurrent_dropout=RECURRENT_DROPOUT,
             use_cudnn=False)(x, mask=mask_input)
    x = Dropout(HEAD_DROPOUT)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model([fused_input, mask_input], outputs, name=name)
    model.compile(optimizer=AdamW(learning_rate=learning_rate, weight_decay=WEIGHT_DECAY),
                  loss=build_loss(), metrics=['accuracy'])
    return model


model_frozen_embedding = build_frozen_embedding_lstm()
model_frozen_embedding.summary()

### Class Weights (Frozen CNN Embedding + LSTM)

This variant computes **its own** `class_weight` rather than inheriting `weight_dict` from the
from-scratch backbone section. That section's `USE_CLASS_WEIGHTS` toggle is an experiment about
one specific failure mode: reweighting on a tiny `BATCH_SIZE=8`, where a single batch can be
dominated by one heavily-weighted example, was a suspected driver of epoch-to-epoch
`val_macro_f1` noise. This model trains at `BATCH_SIZE_FROZEN=32` on small precomputed fused
vectors with no image forward pass per batch, so that interaction doesn't carry over -- the
reason to disable weighting upstream doesn't apply here.

`Drowsy` is the single most safety-critical class in this project -- the state it's least
acceptable to miss -- so this section keeps sklearn's `'balanced'` weights with a `Drowsy` boost
on top, computed from `y_train_fe` (the exact 0-indexed label vector `build_fused_arrays` feeds
the LSTM, not `df_train['level']` -- equivalent here, but this keeps the weights tied to what
actually trains). No `Not Drowsy` dampen is applied, unlike the from-scratch section -- add one
only if `Drowsy` recall comes back weak in the eval cell.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# --- Class weights for the frozen-embedding + LSTM variant (fallback path only) ---
# Only used when USE_BALANCED_SAMPLING is False. The frozen variant trains at BATCH_SIZE_FROZEN=32
# on small precomputed fused vectors, so the small-batch-plus-reweighting concern that motivates
# balanced sampling for the from-scratch model is weaker here -- but sampling is still the primary
# path for consistency, and the final Drowsy operating point comes from the decision-threshold
# cell either way.
DROWSY_LABEL_FROZEN = CLASS_NAMES.index('Drowsy')            # 1
NOT_DROWSY_LABEL_FROZEN = CLASS_NAMES.index('Not Drowsy')    # 0
DROWSY_WEIGHT_BOOST_FROZEN = 1.375

_labels_frozen, _counts_frozen = np.unique(y_train_fe, return_counts=True)
print("Train label distribution (0=Not Drowsy, 1=Drowsy):",
      dict(zip(_labels_frozen.tolist(), _counts_frozen.tolist())))

if USE_BALANCED_SAMPLING:
    weight_dict_frozen = None
    print("USE_BALANCED_SAMPLING=True -> frozen variant also trains with class_weight=None.")
else:
    _classes_frozen = np.unique(y_train_fe)
    _balanced_frozen = compute_class_weight('balanced', classes=_classes_frozen, y=y_train_fe)
    weight_dict_frozen = dict(zip(_classes_frozen.tolist(), _balanced_frozen))
    weight_dict_frozen[DROWSY_LABEL_FROZEN] *= DROWSY_WEIGHT_BOOST_FROZEN
    print(f"Frozen-variant class weights (Drowsy x{DROWSY_WEIGHT_BOOST_FROZEN}): {weight_dict_frozen}")

### Training (Frozen CNN Embedding + LSTM)

Reuses `MacroF1Callback` and the `val_macro_f1`-monitored `EarlyStopping`/`ModelCheckpoint`
pattern from the from-scratch section, for the same cross-variant comparability reasons. Class
weights, though, are **this variant's own** (`weight_dict_frozen`, computed in the cell just
above) rather than the from-scratch section's `weight_dict`: `BATCH_SIZE_FROZEN=32` with only
small precomputed fused vectors per batch removes the small-batch-plus-reweighting interaction
that the from-scratch section's `USE_CLASS_WEIGHTS` experiment is about, so this variant simply
commits to sklearn `'balanced'` weights with a `Drowsy` boost on top. Watch `Not Drowsy`
precision alongside `Drowsy` recall in the eval cell below.

In [ ]:
def make_fused_dataset(X, mask, y, batch_size, training, balanced=False):
    if training and balanced:
        parts = []
        for cls in (0, 1):
            sel = (y == cls)
            parts.append(
                tf.data.Dataset.from_tensor_slices(((X[sel], mask[sel]), y[sel]))
                .shuffle(int(sel.sum()), seed=42, reshuffle_each_iteration=True).repeat()
            )
        ds = tf.data.Dataset.sample_from_datasets(parts, weights=[0.5, 0.5], seed=42)
    else:
        ds = tf.data.Dataset.from_tensor_slices(((X, mask), y))
        if training:
            ds = ds.shuffle(buffer_size=2048, seed=42)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


BATCH_SIZE_FROZEN = 32  # Much larger than the from-scratch variant's BATCH_SIZE=8 -- memory
                         # isn't the constraint it was there, since no images flow through this
                         # pipeline per batch, just small fused vectors.

STEPS_PER_EPOCH_FROZEN = int(np.ceil(len(y_train_fe) / BATCH_SIZE_FROZEN))
train_ds_frozen = make_fused_dataset(X_train_fe, mask_train_fe, y_train_fe, BATCH_SIZE_FROZEN,
                                     training=True, balanced=USE_BALANCED_SAMPLING)
val_ds_frozen = make_fused_dataset(X_val_fe, mask_val_fe, y_val_fe, BATCH_SIZE_FROZEN, training=False)
test_ds_frozen = make_fused_dataset(X_test_fe, mask_test_fe, y_test_fe, BATCH_SIZE_FROZEN, training=False)

macro_f1_cb_frozen = MacroF1Callback(val_ds_frozen)
early_stopping_frozen = EarlyStopping(monitor='val_macro_f1', mode='max', patience=15,
                                       restore_best_weights=True)
checkpoint_frozen = ModelCheckpoint(
    filepath=os.path.join(models_folder, 'best_cnn_lstm_frozen_embedding.keras'),
    monitor='val_macro_f1', mode='max', save_best_only=True,
)

print("🚀 Training CNN+LSTM (frozen CNN embedding + LSTM head)...")
history_frozen_embedding = model_frozen_embedding.fit(
    train_ds_frozen,
    epochs=100,
    steps_per_epoch=STEPS_PER_EPOCH_FROZEN if USE_BALANCED_SAMPLING else None,
    validation_data=val_ds_frozen,
    callbacks=[macro_f1_cb_frozen, early_stopping_frozen, checkpoint_frozen],
    class_weight=None if USE_BALANCED_SAMPLING else weight_dict_frozen,
    verbose=1,
)

### Evaluation (Frozen CNN Embedding + LSTM)

Same `evaluate_variant` helper as the from-scratch section above -- same metrics, same
per-`window_duration_sec` breakdown -- so this variant's numbers are directly comparable to
the from-scratch backbone's (the pre-migration 3-class run measured 32.22% accuracy / 0.3251
macro-F1; no binary run yet) without any change in how either was measured.

In [ ]:
y_true_frozen, y_pred_frozen, per_duration_frozen = evaluate_variant(
    os.path.join(models_folder, 'best_cnn_lstm_frozen_embedding.keras'),
    model_frozen_embedding, test_ds_frozen, df_test, 'CNN+LSTM (frozen CNN embedding)',
)

if 'history_frozen_embedding' in globals():
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history_frozen_embedding.history['accuracy'], label='Training Accuracy')
    plt.plot(history_frozen_embedding.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Model Accuracy (Frozen CNN Embedding)'); plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history_frozen_embedding.history['loss'], label='Training Loss')
    plt.plot(history_frozen_embedding.history['val_loss'], label='Validation Loss')
    plt.title('Model Loss (Frozen CNN Embedding)'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Operating point for the frozen-CNN-embedding + LSTM variant. Uses its own val/test datasets
# (small precomputed fused vectors, BATCH_SIZE_FROZEN) -- otherwise identical analysis.
t_star_frozen = operating_point_analysis(
    'CNN+LSTM (frozen CNN embedding)',
    os.path.join(models_folder, 'best_cnn_lstm_frozen_embedding.keras'),
    model_frozen_embedding, val_ds_frozen, test_ds_frozen, test_df=df_test,
)


## Ensemble: from-scratch ⊕ frozen-embedding

Both variants are evaluated on the **same `df_test` rows in the same order** (`test_ds` is
unshuffled; `X_test_fe` is built from `df_test` in order), so their `p(Drowsy)` vectors align
1:1 and can be averaged. Late fusion of two models that see genuinely different information — an
end-to-end conv stack over image sequences vs. a frozen single-frame embedding + geometric
features — often beats either alone when their errors are not fully correlated. The Part-2
threshold sweep is then run on the averaged probabilities.

(A three-way ensemble that also folds in `07`'s single-frame CNN is a follow-up: `07`'s fold-0
test subjects are not guaranteed to match this notebook's, so it needs explicit subject
alignment first.)


In [ ]:
# Average p(Drowsy) of the from-scratch and frozen-embedding models, then pick an operating point.
_m_s = _load_for_eval(os.path.join(models_folder, 'best_cnn_lstm_scratch.keras'), model_scratch)
_m_f = _load_for_eval(os.path.join(models_folder, 'best_cnn_lstm_frozen_embedding.keras'), model_frozen_embedding)

yv_s, pv_s = _p_drowsy(_m_s, val_ds)
yv_f, pv_f = _p_drowsy(_m_f, val_ds_frozen)
yt_s, pt_s = _p_drowsy(_m_s, test_ds)
yt_f, pt_f = _p_drowsy(_m_f, test_ds_frozen)
assert np.array_equal(yv_s, yv_f) and np.array_equal(yt_s, yt_f), \
    "row order mismatch between the two variants -- can't average their probabilities"

pv_ens, pt_ens = 0.5 * (pv_s + pv_f), 0.5 * (pt_s + pt_f)

rows = _sweep(yv_s, pv_ens)
star, f1_star, note = _pick(rows, DROWSY_RECALL_FLOOR)
print("=" * 78)
print(f"ENSEMBLE (0.5 * scratch + 0.5 * frozen)  chosen t* (val) = {star['t']:.2f}  [{note}]")
print(f"  val @ t*: macro-F1 {star['macro_f1']:.4f}  Drowsy P {star['drowsy_p']:.3f}  R {star['drowsy_r']:.3f}")
base_e = _report_at(yt_s, pt_ens, 0.50, "ENSEMBLE TEST @ argmax")
tuned_e = _report_at(yt_s, pt_ens, star['t'], "ENSEMBLE TEST @ chosen t*")
print("\nsingle-model test macro-F1 @ their own t*: "
      "(see the two operating-point cells above) -- ensemble should be >= the better of them")

with open(os.path.join(models_folder, 'cnn_lstm_ensemble.threshold.json'), 'w') as fh:
    json.dump(dict(members=['best_cnn_lstm_scratch.keras', 'best_cnn_lstm_frozen_embedding.keras'],
                   weights=[0.5, 0.5], drowsy_index=DROWSY_IDX, threshold=star['t'],
                   test_at_threshold=tuned_e, test_at_argmax=base_e), fh, indent=2)
print("wrote", os.path.join(models_folder, 'cnn_lstm_ensemble.threshold.json'))


## Reading This Result Alongside the Other Models

**Current status: MobileNetV2 is disabled (see above); from-scratch is the backbone this notebook
is actually pursuing going forward.** Across every real run so far, MobileNetV2 has overfit more
severely than the from-scratch backbone, not just scored lower — most recently 95%+ train
accuracy against a `val_macro_f1` ceiling around 0.36 (vs. from-scratch's 0.42 peak, 0.3296 test
macro-F1 in the same run). This isn't one bad run: 07's single-frame MobileNetV2 collapsed
similarly (16.81% accuracy, zero recall on one class), and this notebook's own earlier
pre-fusion run also had MobileNetV2 underperforming from-scratch on the full subject pool. The
`alpha=1.0`/`IMG_SIZE_MOBILENET=128` fix (vs. 07's `alpha=0.35`/96×96) was a real, reasoned
attempt at the resolution/alpha mismatch diagnosed in `notebook/CLAUDE.md`'s "Cheaper options" —
it wasn't the answer; frozen ImageNet features plus a from-scratch conv stack trained specifically
for this task's small, tightly-cropped face images just doesn't transfer as well here as training
that stack from scratch does.

**Pre-migration 3-class record (no binary run yet).** The from-scratch backbone's last 3-class
run (`AdamW` + `recurrent_dropout` + `class_weight=None`) measured 35.04% test accuracy, 0.3296
macro-F1 — a real improvement over the same backbone's prior run without those changes (29.83% /
0.2801), and close to but never clearly beating 07's single-frame CNN (36.67% / 0.3614). That
last comparison is the one that still matters after the binary switch: the temporal-context
premise was never validated against the single-frame CNN in 3-class, and collapsing to two
classes doesn't answer it either. `val_macro_f1` still peaked early (epoch 5, 0.4182) and declined afterward as
`ReduceLROnPlateau` cut the learning rate down through the rest of the run — direct, within-run
evidence that a *lower* learning rate didn't help here either (val_macro_f1 at the lowest LR,
6.25e-06, was ~0.22-0.23, worse than at every higher LR tried earlier in the same run), on top of
the cross-run evidence already gathered against that lever.

- `07_cnn_training.ipynb`'s single-frame CNN (36.67% accuracy, 0.3614 macro-F1, from-scratch;
  16.81% accuracy, MobileNetV2) — this run's from-scratch CNN+LSTM (35.04%/0.3296) is now close
  to but still hasn't clearly beaten 07's single-frame result, so the temporal-context premise
  still isn't validated, even with the fusion/regularization changes.
- The per-duration breakdown (see the from-scratch eval cell above) is the direct test of whether
  more temporal context is paying for itself at all — check that before assuming duration is
  helping.

**Not built here, worth doing next depending on what further runs show:**
- A subject-fold cross-validation diagnostic (mirroring 07's `RUN_CROSS_VALIDATION`) — see the
  "Splitting the Dataset" markdown cell's note on the small, fixed 234-window validation pool
  being a plausible independent source of `val_macro_f1` noise, on top of the LR/overfitting
  question above.
- A learning-rate *schedule* change (e.g. cosine decay with warm restarts) instead of
  `ReduceLROnPlateau`'s monotonic decay — under discussion, not yet decided or implemented.
- If MobileNetV2 is ever revisited, see `notebook/CLAUDE.md`'s "Cheaper options considered
  alongside the full CNN+LSTM build" for lower-data-cost fallbacks already discussed (a
  late-fusion ensemble of RandomForest/Dense NN/CNN, or feeding 07's frozen CNN embedding into
  the existing geometric LSTM instead of training a CNN from scratch inside the recurrent loop).
